# Feature Space Lab v2: максимально широкий пайплайн признаков

Выходные файлы сохраняются в:

```text
<repo_root>/prepared_data/
```

Что добавлено по сравнению с прошлой версией:

1. Более аккуратное определение корня проекта.
2. Больше row-level sparse/meta признаков.
3. Block statistics по нескольким разбиениям колонок.
4. Top-k статистики по разным рейтингам признаков.
5. Frequency encoding и OOF target encoding.
6. Автоматический поиск `peak flags`.
7. Pairwise interactions для топовых признаков.
8. PCA, Quantile-PCA, SVD по nonzero-mask.
9. KMeans-сегменты.
10. Adversarial validation train-vs-test и списки drifty/stable features.
11. Model-based OOF meta-features: rough LightGBM / Logistic predictions.
12. Noise candidate detection по OOF-предсказаниям.
13. Расширенный `feature_sets.json`, совместимый с моделями дальше.
14. Бенчмарк feature sets одним и тем же LightGBM.


## 0. Настройки запуска

In [1]:
from __future__ import annotations

import gc
import json
import math
import os
import sys
import time
import warnings
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.model_selection import StratifiedKFold, train_test_split
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = True
except Exception:
    HAS_STRATIFIED_GROUP_KFOLD = False

from sklearn.metrics import roc_auc_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.cluster import MiniBatchKMeans
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

try:
    import scipy.sparse as sp
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except Exception:
    HAS_LIGHTGBM = False

RANDOM_STATE = 42
N_SPLITS = 5
N_JOBS = max(1, (os.cpu_count() or 4) - 1)

# Тяжёлые блоки. Для финального прогона лучше оставить True.
RUN_MUTUAL_INFO = True
RUN_LGB_IMPORTANCE = True
RUN_ADVERSARIAL_VALIDATION = True
RUN_TARGET_ENCODING = True
RUN_INTERACTIONS = True
RUN_PCA = True
RUN_SVD = True
RUN_KMEANS = True
RUN_MODEL_META_FEATURES = True
RUN_BENCHMARK = True

# Размеры/лимиты. Их можно увеличивать, если хочется совсем пожечь ноутбук.
MI_SAMPLE_SIZE = 120_000
LGB_IMPORTANCE_FOLDS = 5
TARGET_ENCODING_TOP_N = 80
FREQUENCY_ENCODING_TOP_N = 120
INTERACTION_TOP_N = 32
PCA_TOP_N = 700
PCA_N_COMPONENTS = 48
QUANTILE_PCA_TOP_N = 350
QUANTILE_PCA_N_COMPONENTS = 32
SVD_TOP_N = 1000
SVD_N_COMPONENTS = 64
KMEANS_CLUSTERS = 24
BENCHMARK_MAX_FEATURE_SETS = None  # None = все ключевые наборы

np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

print("Python:", sys.version)
print("CPU count:", os.cpu_count(), "N_JOBS:", N_JOBS)
print("LightGBM:", HAS_LIGHTGBM)
print("SciPy:", HAS_SCIPY)
print("StratifiedGroupKFold:", HAS_STRATIFIED_GROUP_KFOLD)

Python: 3.14.3 (main, Feb  3 2026, 15:32:20) [Clang 17.0.0 (clang-1700.6.3.2)]
CPU count: 14 N_JOBS: 13
LightGBM: True
SciPy: True
StratifiedGroupKFold: True


## 1. Корень проекта, входы и выходы

Так как ноутбук будет лежать в `notebooks/`, `Path.cwd()` может быть либо корнем проекта, либо папкой `notebooks`. Поэтому корень определяем эвристически: ищем папки/файлы проекта и поднимаемся на уровень выше при необходимости.

In [2]:
def looks_like_repo_root(p: Path) -> bool:
    markers = ["data", "raw_data", "prepared_data", "processed_data", ".git"]
    return any((p / m).exists() for m in markers) or (p / "train.csv").exists()


def infer_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for c in candidates:
        if looks_like_repo_root(c):
            return c.resolve()
    return cwd


def find_first_existing_file(names: Sequence[str], search_dirs: Sequence[Path]) -> Optional[Path]:
    for d in search_dirs:
        for name in names:
            p = d / name
            if p.exists():
                return p.resolve()
    return None

PROJECT_ROOT = infer_project_root()
OUTPUT_DIR = PROJECT_ROOT / "prepared_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_DIRS = [
    PROJECT_ROOT,
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "raw_data",
    PROJECT_ROOT / "ml_csvs",
    PROJECT_ROOT / "csv",
    PROJECT_ROOT / "input",
    PROJECT_ROOT.parent,
    PROJECT_ROOT.parent / "data",
    PROJECT_ROOT.parent / "raw_data",
    Path("/mnt/data/ml_csvs"),
    Path("/mnt/data"),
]

TRAIN_PATH = find_first_existing_file(["train.csv"], SEARCH_DIRS)
TEST_PATH = find_first_existing_file(["test.csv"], SEARCH_DIRS)
SAMPLE_SUB_PATH = find_first_existing_file(["sample_submission.csv", "submission.csv"], SEARCH_DIRS)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH:", TEST_PATH)
print("SAMPLE_SUB_PATH:", SAMPLE_SUB_PATH)

assert TRAIN_PATH is not None, "Не найден train.csv. Положи его в корень/data/raw_data или поправь SEARCH_DIRS."
assert TEST_PATH is not None, "Не найден test.csv. Положи его в корень/data/raw_data или поправь SEARCH_DIRS."

PROJECT_ROOT: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton
OUTPUT_DIR: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/prepared_data
TRAIN_PATH: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/data/train.csv
TEST_PATH: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/data/test.csv
SAMPLE_SUB_PATH: None


## 2. Загрузка CSV с экономными типами

Все `feature_*` читаем как `float32`: для бустингов этого достаточно, зато память экономится примерно в 2 раза по сравнению с `float64`.

In [3]:
def build_dtype_map(csv_path: Path) -> Dict[str, str]:
    cols = pd.read_csv(csv_path, nrows=0).columns.tolist()
    dtype_map: Dict[str, str] = {}
    for c in cols:
        if c == "index":
            dtype_map[c] = "int64"
        elif c == "target":
            dtype_map[c] = "int8"
        elif c.startswith("feature_"):
            dtype_map[c] = "float32"
    return dtype_map


def read_main_csv(path: Path) -> pd.DataFrame:
    t0 = time.time()
    df = pd.read_csv(path, dtype=build_dtype_map(path))
    mem_gb = df.memory_usage(deep=True).sum() / 1024**3
    print(f"Loaded {path.name}: shape={df.shape}, memory={mem_gb:.2f} GB, time={time.time() - t0:.1f}s")
    return df

train = read_main_csv(TRAIN_PATH)
test = read_main_csv(TEST_PATH)

sample_submission = None
if SAMPLE_SUB_PATH is not None:
    sample_submission = pd.read_csv(SAMPLE_SUB_PATH)
    print("sample_submission:", sample_submission.shape)

ID_COL = "index"
TARGET_COL = "target"
feature_cols = [c for c in train.columns if c.startswith("feature_") and c in test.columns]

assert TARGET_COL in train.columns, "В train.csv не найден target."
assert ID_COL in train.columns and ID_COL in test.columns, "В train/test должен быть index."
assert len(feature_cols) > 0, "Не найдены feature_* колонки."

print("Feature count:", len(feature_cols))

Loaded train.csv: shape=(247972, 1369), memory=1.26 GB, time=9.9s
Loaded test.csv: shape=(106274, 1368), memory=0.54 GB, time=4.1s
Feature count: 1367


## 3. Базовая диагностика

Смотрим размер, дисбаланс, пропуски, нули и кардинальности. Это влияет на все решения дальше: какие признаки удалять, какие кодировки делать, нужен ли `imputer`, как строить валидацию.

In [4]:
display(train.head())
display(test.head())

print("Train shape:", train.shape)
print("Test shape:", test.shape)

vc = train[TARGET_COL].value_counts().sort_index()
print("\nTarget counts:")
print(vc)
print("Target rate:", float(train[TARGET_COL].mean()))
print("Class imbalance neg/pos:", float(vc.get(0, 0) / max(vc.get(1, 1), 1)))

missing_train = int(train[feature_cols].isna().sum().sum())
missing_test = int(test[feature_cols].isna().sum().sum())
print("\nTotal missing values in train features:", missing_train)
print("Total missing values in test features:", missing_test)

zero_frac_train = train[feature_cols].eq(0).mean()
zero_frac_test = test[feature_cols].eq(0).mean()
nunique_train = train[feature_cols].nunique(dropna=False)

feature_profile = pd.DataFrame({
    "zero_frac_train": zero_frac_train,
    "zero_frac_test": zero_frac_test,
    "zero_frac_delta": (zero_frac_train - zero_frac_test).abs(),
    "nunique_train": nunique_train,
    "mean_train": train[feature_cols].mean(),
    "mean_test": test[feature_cols].mean(),
    "std_train": train[feature_cols].std(),
    "std_test": test[feature_cols].std(),
    "min_train": train[feature_cols].min(),
    "max_train": train[feature_cols].max(),
})
feature_profile["mean_abs_delta"] = (feature_profile["mean_train"] - feature_profile["mean_test"]).abs()
feature_profile["std_ratio_train_test"] = feature_profile["std_train"] / (feature_profile["std_test"].replace(0, np.nan))

print("\nZero fraction summary:")
display(feature_profile["zero_frac_train"].describe())
print("\nMost sparse features:")
display(feature_profile.sort_values("zero_frac_train", ascending=False).head(20))
print("\nLowest cardinality features:")
display(feature_profile.sort_values("nunique_train").head(20))

,index,target,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40,feature_41,feature_42,feature_43,feature_44,feature_45,feature_46,feature_47,feature_48,feature_49,feature_50,feature_51,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,...,feature_1307,feature_1308,feature_1309,feature_1310,feature_1311,feature_1312,feature_1313,feature_1314,feature_1315,feature_1316,feature_1317,feature_1318,feature_1319,feature_1320,feature_1321,feature_1322,feature_1323,feature_1324,feature_1325,feature_1326,feature_1327,feature_1328,feature_1329,feature_1330,feature_1331,feature_1332,feature_1333,feature_1334,feature_1335,feature_1336,feature_1337,feature_1338,feature_1339,feature_1340,feature_1341,feature_1342,feature_1343,feature_1344,feature_1345,feature_1346,feature_1347,feature_1348,feature_1349,feature_1350,feature_1351,feature_1352,feature_1353,feature_1354,feature_1355,feature_1356,feature_1357,feature_1358,feature_1359,feature_1360,feature_1361,feature_1362,feature_1363,feature_1364,feature_1365,feature_1366
0,239134,0,0.5,0.5,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.680302,0.767927,0.917902,0.527302,0.428348,0.466657,0.548376,0.815871,0.806104,0.729981,0.247545,0.604350,0.319196,0.717840,0.727683,0.769678,0.685146,0.582802,0.680368,0.890976,0.719449,0.635555,0.622619,0.489604,0.520638,0.460173,0.440582,0.524772,0.447167,0.480903,0.444122,0.735702,0.902803,0.395097,0.736835,0.490412,0.596536,0.904454,0.732110,0.0,0.011896,5.437209,39.134056,0.673629,1.419289,5.696633,15.822751,0.000508,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
1,234708,0,0.5,0.5,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.680302,0.767927,0.917902,0.527302,0.428348,0.466657,0.548376,0.815871,0.806104,0.729981,0.247545,0.604350,0.319196,0.717840,0.727683,0.769678,0.685146,0.582802,0.680368,0.890976,0.719449,0.635555,0.622619,0.489604,0.520638,0.460173,0.440582,0.524772,0.447167,0.480903,0.444122,0.735702,0.902803,0.395097,0.736835,0.490412,0.596536,0.904454,0.732110,0.0,0.011896,5.437209,39.134056,0.673629,1.419289,5.696633,15.822751,0.000508,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0
2,268300,0,0.5,0.5,1.0,1.0,1.0,3.0,1.0,4.0,0.0,9.0,0.794758,2.471037,1.010160,0.288224,0.051471,0.154412,0.930185,0.970914,1.047707,1.046696,0.302190,0.970542,0.429787,0.670482,0.659651,0.726351,0.667095,0.609957,0.654339,0.790576,0.674854,0.593292,0.494033,0.494028,0.336947,0.071942,0.071942,0.812248,0.071942,0.140814,0.075436,0.596417,0.845669,0.103440,0.598182,0.475128,0.501427,0.995018,0.709474,0.0,0.000000,522.000000,28.000000,0.000000,12.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,283077,0,0.5,0.5,1.0,1.0,1.0,3.0,1.0,0.0,3.0,8.0,0.917668,0.558128,0.972966,0.213058,0.098592,0.072165,1.113402,0.987482,0.986724,0.986697,0.291172,1.117952,0.455936,0.638321,0.609359,0.696601,0.616842,0.494184,0.629273,0.760018,0.614402,0.492812,0.405435,0.379545,0.270000,0.200000,0.100000,1.207500,0.100000,0.180000,0.090000,0.669705,1.043027

,index,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40,feature_41,feature_42,feature_43,feature_44,feature_45,feature_46,feature_47,feature_48,feature_49,feature_50,feature_51,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,feature_58,...,feature_1307,feature_1308,feature_1309,feature_1310,feature_1311,feature_1312,feature_1313,feature_1314,feature_1315,feature_1316,feature_1317,feature_1318,feature_1319,feature_1320,feature_1321,feature_1322,feature_1323,feature_1324,feature_1325,feature_1326,feature_1327,feature_1328,feature_1329,feature_1330,feature_1331,feature_1332,feature_1333,feature_1334,feature_1335,feature_1336,feature_1337,feature_1338,feature_1339,feature_1340,feature_1341,feature_1342,feature_1343,feature_1344,feature_1345,feature_1346,feature_1347,feature_1348,feature_1349,feature_1350,feature_1351,feature_1352,feature_1353,feature_1354,feature_1355,feature_1356,feature_1357,feature_1358,feature_1359,feature_1360,feature_1361,feature_1362,feature_1363,feature_1364,feature_1365,feature_1366
0,194357,0.5,0.5,1.0,1.0,1.0,1.0,1.0,3.0,0.0,6.0,0.968182,0.714778,1.265306,0.393116,0.152174,0.152174,0.880435,0.999675,0.956950,0.898636,0.215609,0.646174,0.354864,0.864612,0.900893,0.922357,0.887152,0.835618,0.899584,0.908295,0.900525,0.877328,0.878003,0.851101,0.189474,0.105263,0.105263,0.847368,0.105263,0.189474,0.284211,0.729507,0.926831,0.234820,0.965565,0.848896,0.385638,1.034000,0.934553,0.0,0.0,0.0,341.0,0.0,14.0,0.0,10.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,313222,0.5,0.5,1.0,1.0,0.0,2.0,2.0,2.0,0.0,6.0,0.638919,2.277712,0.972322,0.645833,0.875000,0.875000,0.562500,1.028136,0.861230,0.592205,0.468825,0.561151,0.592105,0.801061,0.749033,0.809998,0.751962,0.626032,0.722613,0.906852,0.718453,0.759986,0.750598,0.542589,0.818182,0.909091,0.909091,0.522727,0.909091,0.818182,0.818182,0.610724,0.910857,0.144266,0.950112,0.573724,0.312500,0.966167,0.750587,0.0,0.0,1.0,9.0,0.0,9.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,321873,0.5,0.5,1.0,1.0,0.0,3.0,1.0,1.0,0.0,5.0,0.680302,0.767927,0.917902,0.527302,0.428348,0.466657,0.548376,0.815871,0.806104,0.729981,0.247545,0.604350,0.319196,0.717840,0.727683,0.769678,0.685146,0.582802,0.680368,0.890976,0.719449,0.635555,0.622619,0.489604,0.520638,0.460173,0.440582,0.524772,0.447167,0.480903,0.444122,0.735702,0.902803,0.395097,0.736835,0.490412,0.596536,0.904454,0.732110,0.0,0.0,0.0,229.0,0.0,0.0,0.0,6.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,118689,0.5,0.5,1.0,1.0,1.0,3.0,2.0,3.0,0.0,9.0,0.684927,0.852196,0.959507,0.218927,0.118644,0.118644,0.610170,1.012567,1.025495,0.930575,0.387470,0.982014,0.470395,0.479265,0.554413,0.628606,0.482222,0.372899,0.472783,0.889012,0.570063,0.497831,0.452277,0.293005,0.446281,0.082645,0.082645,0.712810,0.082645,0.074380,0.074380,0.863935,0.796311,0.217637,0.245532,0.299714,0.241228,0.975725,0.587085,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,

Train shape: (247972, 1369)
Test shape: (106274, 1368)

Target counts:
target
0    244626
1      3346
Name: count, dtype: int64
Target rate: 0.013493458938912458
Class imbalance neg/pos: 73.10998206814106

Total missing values in train features: 0
Total missing values in test features: 0

Zero fraction summary:


count    1367.000000
mean        0.733958
std         0.259026
min         0.000000
25%         0.704959
50%         0.844144
75%         0.903239
max         1.000000
Name: zero_frac_train, dtype: float64


Most sparse features:


,zero_frac_train,zero_frac_test,zero_frac_delta,nunique_train,mean_train,mean_test,std_train,std_test,min_train,max_train,mean_abs_delta,std_ratio_train_test
feature_49,1.000000,1.000000,0.000000,1,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000e+00,NaN
feature_1064,1.000000,1.000000,0.000000,1,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000e+00,NaN
feature_1057,1.000000,1.000000,0.000000,1,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000e+00,NaN
feature_1363,0.965258,0.965344,0.000086,2,0.034742,0.034656,0.183126,0.182907,0.0,1.000000,8.612499e-05,1.001194
feature_697,0.903796,0.904774,0.000979,2,0.000074,0.000073,0.000227,0.000226,0.0,0.000769,7.526760e-07,1.004580
feature_698,0.903796,0.904774,0.000979,2,0.000029,0.000029,0.000090,0.000089,0.0,0.000304,2.975048e-07,1.004580
feature_536,0.903796,0.904774,0.000979,2,0.000459,0.000455,0.001408,0.001401,0.0,0.004774,4.672707e-06,1.004580
feature_987,0.903796,0.904765,0.000969,2,0.000409,0.000424,0.001255,0.006260,0.0,0.004256,1.465282e-05,0.200504
feature_532,0.903796,0.904774,0.000979,2,0.000284,0.000281,0.000870,0.000866,0.0,0.002952,2.889399e-06,1.004580
feature_507,0.903796,0.904774,0.000979,2,0.000128,0.000127,0.000394,0.000392,0.0,0.001335,1.306777e-06,1.004580



Lowest cardinality features:


,zero_frac_train,zero_frac_test,zero_frac_delta,nunique_train,mean_train,mean_test,std_train,std_test,min_train,max_train,mean_abs_delta,std_ratio_train_test
feature_0,0.000000,0.000000,0.000000,1,0.500000,0.500000,0.000000,0.000000,0.5,0.500000,0.000000e+00,NaN
feature_1064,1.000000,1.000000,0.000000,1,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000e+00,NaN
feature_49,1.000000,1.000000,0.000000,1,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000e+00,NaN
feature_1057,1.000000,1.000000,0.000000,1,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000e+00,NaN
feature_2,0.000000,0.000000,0.000000,1,1.000000,1.000000,0.000000,0.000000,1.0,1.000000,0.000000e+00,NaN
feature_3,0.000000,0.000000,0.000000,1,1.000000,1.000000,0.000000,0.000000,1.0,1.000000,0.000000e+00,NaN
feature_1,0.000000,0.000000,0.000000,1,0.500000,0.500000,0.000000,0.000000,0.5,0.500000,0.000000e+00,NaN
feature_504,0.903796,0.904774,0.000979,2,0.000003,0.000003,0.000010,0.000010,0.0,0.000035,3.460946e-08,1.004580
feature_240,0.863259,0.865527,0.002268,2,0.027039,0.026591,0.067938,0.067461,0.0,0.197739,4.484914e-04,1.007073
feature_246,0.863259,0.865527,0.002268,2,0.024196,0.023794,0.060794,0.060367,0.0,0.176945,4.013255e-04,1.007073


## 4. Дубли, exact train/test overlap и честные folds

Если одинаковые строки попадают в разные folds, CV становится завышенным. Поэтому строим `group = hash(feature_vector)` и используем `StratifiedGroupKFold`, если он доступен.

Exact overlap между train и test отдельно сохраняем как файл для возможного postprocessing: если test-строка точно совпадает с train-строкой, у нас есть эмпирический `target_mean` этой группы.

In [5]:
from pandas.util import hash_pandas_object

t0 = time.time()
train_feature_hash = hash_pandas_object(train[feature_cols], index=False).astype("uint64")
test_feature_hash = hash_pandas_object(test[feature_cols], index=False).astype("uint64")
print(f"Hashes computed in {time.time() - t0:.1f}s")

train_dup_counts = train_feature_hash.value_counts()
test_dup_counts = test_feature_hash.value_counts()

print("Train rows in duplicated feature vectors:", int(train_dup_counts[train_dup_counts > 1].sum()))
print("Train duplicated feature-vector groups:", int((train_dup_counts > 1).sum()))
print("Test rows in duplicated feature vectors:", int(test_dup_counts[test_dup_counts > 1].sum()))
print("Test duplicated feature-vector groups:", int((test_dup_counts > 1).sum()))

train_hash_target_stats = (
    pd.DataFrame({"feature_hash": train_feature_hash.values, TARGET_COL: train[TARGET_COL].values})
    .groupby("feature_hash")[TARGET_COL]
    .agg(["count", "sum", "mean"])
    .reset_index()
)
conflict_hash_groups = train_hash_target_stats[(train_hash_target_stats["count"] > 1) & (train_hash_target_stats["mean"] > 0) & (train_hash_target_stats["mean"] < 1)]
print("Conflict duplicate hash groups:", len(conflict_hash_groups))
display(conflict_hash_groups.sort_values("count", ascending=False).head(20))

# train/test overlap
train_hash_stats = train_hash_target_stats.rename(columns={"count": "train_count", "sum": "train_positive_count", "mean": "known_target_mean"})
test_hash_df = pd.DataFrame({ID_COL: test[ID_COL].values, "feature_hash": test_feature_hash.values})
test_overlap = test_hash_df.merge(train_hash_stats, on="feature_hash", how="inner")
print("Exact train/test overlap rows:", len(test_overlap))
display(test_overlap.head())

test_overlap[[ID_COL, "feature_hash", "known_target_mean", "train_count", "train_positive_count"]].to_csv(
    OUTPUT_DIR / "test_train_exact_overlap.csv", index=False
)
print("Saved:", OUTPUT_DIR / "test_train_exact_overlap.csv")

Hashes computed in 1.1s
Train rows in duplicated feature vectors: 18746
Train duplicated feature-vector groups: 7139
Test rows in duplicated feature vectors: 4715
Test duplicated feature-vector groups: 1545
Conflict duplicate hash groups: 93


,feature_hash,count,sum,mean
138505,10810876695524086779,364,6,0.016484
34780,2719412945535468795,316,3,0.009494
192163,15024679413994247915,315,2,0.006349
83832,6518861788221113675,161,3,0.018634
46530,3627506730639429941,108,2,0.018519
126225,9844721106959895500,48,1,0.020833
165237,12907178941986223151,45,2,0.044444
208577,16284551266699955007,44,2,0.045455
95599,7436395924665408389,44,2,0.045455
328,26539138358087432,43,1,0.023256


Exact train/test overlap rows: 8128


,index,feature_hash,train_count,train_positive_count,known_target_mean
0,132875,11246226303984376264,1,0,0.000000
1,280898,13864781300107392152,24,0,0.000000
2,216290,10003429060184137555,1,1,1.000000
3,199917,6339280815843837279,1,0,0.000000
4,289908,2719412945535468795,316,3,0.009494


Saved: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/prepared_data/test_train_exact_overlap.csv


In [6]:
y = train[TARGET_COL].astype("int8").values


def make_cv_splits(y: np.ndarray, groups: Optional[np.ndarray] = None, n_splits: int = 5, random_state: int = 42):
    if groups is not None and HAS_STRATIFIED_GROUP_KFOLD:
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        return list(cv.split(np.zeros(len(y)), y, groups=groups))
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    return list(cv.split(np.zeros(len(y)), y))

cv_splits = make_cv_splits(y, groups=train_feature_hash.values, n_splits=N_SPLITS, random_state=RANDOM_STATE)
print("CV folds:", len(cv_splits))
for i, (tr_idx, va_idx) in enumerate(cv_splits):
    print(i, "train", len(tr_idx), "valid", len(va_idx), "valid target rate", y[va_idx].mean())

fold_df = pd.DataFrame({ID_COL: train[ID_COL].values, "fold": -1, "feature_hash": train_feature_hash.values})
for fold, (_, va_idx) in enumerate(cv_splits):
    fold_df.loc[va_idx, "fold"] = fold
fold_df.to_csv(OUTPUT_DIR / "cv_folds_group_hash.csv", index=False)
print("Saved:", OUTPUT_DIR / "cv_folds_group_hash.csv")

CV folds: 5
0 train 198376 valid 49596 valid target rate 0.013509153964029358
1 train 198378 valid 49594 valid target rate 0.013489535024398112
2 train 198378 valid 49594 valid target rate 0.013489535024398112
3 train 198378 valid 49594 valid target rate 0.013489535024398112
4 train 198378 valid 49594 valid target rate 0.013489535024398112
Saved: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/prepared_data/cv_folds_group_hash.csv


## 5. Базовая очистка признаков

Удаляем только то, что почти точно бесполезно: константные и квазиконстантные признаки. Агрессивное удаление выбросов не делаем: в таких данных редкие большие значения часто являются сигналом.

In [7]:
LOW_VARIANCE_TOP_SHARE = 0.995

low_variance_cols: List[str] = []
for col in feature_cols:
    top_share = train[col].value_counts(normalize=True, dropna=False).iloc[0]
    if top_share >= LOW_VARIANCE_TOP_SHARE:
        low_variance_cols.append(col)

print("Low-variance cols:", len(low_variance_cols))
print(low_variance_cols[:50])

raw_feature_cols = [c for c in feature_cols if c not in low_variance_cols]
X_train_raw_clean = train[raw_feature_cols].copy()
X_test_raw_clean = test[raw_feature_cols].copy()
X_test_raw_clean = X_test_raw_clean.reindex(columns=X_train_raw_clean.columns, fill_value=0)

# на всякий случай приводим к float32
for df in [X_train_raw_clean, X_test_raw_clean]:
    float_cols = df.select_dtypes(include=["float64"]).columns
    if len(float_cols):
        df[float_cols] = df[float_cols].astype("float32")

print("Raw clean shapes:", X_train_raw_clean.shape, X_test_raw_clean.shape)

Low-variance cols: 7
['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_49', 'feature_1057', 'feature_1064']
Raw clean shapes: (247972, 1360) (106274, 1360)


## 6. Базовые рейтинги признаков: корреляция, single-feature AUC, MI

Корреляция и AUC по одному признаку слабые для нелинейных задач, но они дают полезный первый рейтинг. Mutual information ловит часть нелинейных зависимостей, но считается дольше, поэтому можно сэмплировать строки.

In [8]:
def safe_abs_corr_with_target(df: pd.DataFrame, y: np.ndarray) -> pd.Series:
    scores = {}
    y_float = y.astype("float32")
    y_std = y_float.std()
    for col in df.columns:
        x = df[col].values.astype("float32")
        x_std = np.nanstd(x)
        if x_std == 0 or not np.isfinite(x_std):
            scores[col] = 0.0
            continue
        c = np.nanmean((x - np.nanmean(x)) * (y_float - y_float.mean())) / (x_std * y_std + 1e-12)
        scores[col] = abs(float(c)) if np.isfinite(c) else 0.0
    return pd.Series(scores, name="abs_corr")


def single_feature_auc_scores(df: pd.DataFrame, y: np.ndarray, cols: Optional[List[str]] = None) -> pd.Series:
    if cols is None:
        cols = list(df.columns)
    out = {}
    for col in cols:
        x = df[col].values
        if np.nanstd(x) == 0:
            out[col] = 0.5
            continue
        try:
            auc = roc_auc_score(y, x)
            out[col] = max(auc, 1 - auc)  # направление признака может быть обратным
        except Exception:
            out[col] = 0.5
    return pd.Series(out, name="single_feature_auc")


t0 = time.time()
abs_corr = safe_abs_corr_with_target(X_train_raw_clean, y)
single_auc = single_feature_auc_scores(X_train_raw_clean, y)
print("Correlation/AUC computed in", round(time.time() - t0, 1), "s")

display(abs_corr.sort_values(ascending=False).head(20))
display(single_auc.sort_values(ascending=False).head(20))

Correlation/AUC computed in 18.0 s


feature_1094    0.020181
feature_520     0.018198
feature_337     0.017494
feature_525     0.017160
feature_1193    0.016612
feature_890     0.016557
feature_1087    0.015832
feature_4       0.015316
feature_1122    0.015188
feature_1132    0.014610
feature_13      0.013815
feature_1110    0.013611
feature_1134    0.013575
feature_1130    0.013510
feature_41      0.013425
feature_341     0.012628
feature_34      0.012572
feature_1195    0.012234
feature_1202    0.012036
feature_43      0.011912
Name: abs_corr, dtype: float64

feature_377     0.554352
feature_1094    0.550985
feature_363     0.550945
feature_283     0.550416
feature_72      0.550066
feature_282     0.549218
feature_823     0.547728
feature_829     0.547550
feature_819     0.547473
feature_294     0.547447
feature_366     0.547280
feature_1034    0.546554
feature_355     0.546470
feature_816     0.546089
feature_821     0.546088
feature_386     0.544084
feature_481     0.543990
feature_99      0.542161
feature_195     0.541320
feature_1087    0.541136
Name: single_feature_auc, dtype: float64

In [9]:
mi_scores = pd.Series(0.0, index=X_train_raw_clean.columns, name="mutual_info")
if RUN_MUTUAL_INFO:
    t0 = time.time()
    rng = np.random.default_rng(RANDOM_STATE)
    if len(y) > MI_SAMPLE_SIZE:
        # стратифицированный сэмпл: берём все positives и часть negatives
        pos_idx = np.flatnonzero(y == 1)
        neg_idx = np.flatnonzero(y == 0)
        neg_take = max(1, MI_SAMPLE_SIZE - len(pos_idx))
        neg_sample = rng.choice(neg_idx, size=min(neg_take, len(neg_idx)), replace=False)
        mi_idx = np.concatenate([pos_idx, neg_sample])
        rng.shuffle(mi_idx)
    else:
        mi_idx = np.arange(len(y))

    X_mi = X_train_raw_clean.iloc[mi_idx].replace([np.inf, -np.inf], np.nan).fillna(0)
    y_mi = y[mi_idx]
    mi = mutual_info_classif(
        X_mi,
        y_mi,
        discrete_features=False,
        random_state=RANDOM_STATE,
        n_neighbors=3,
    )
    mi_scores = pd.Series(mi, index=X_train_raw_clean.columns, name="mutual_info")
    print(f"MI computed on {len(mi_idx)} rows in {time.time() - t0:.1f}s")
    display(mi_scores.sort_values(ascending=False).head(20))
else:
    print("RUN_MUTUAL_INFO=False")

MI computed on 120000 rows in 251.6s


feature_1140    0.020942
feature_1098    0.018884
feature_1082    0.018825
feature_1168    0.018033
feature_1138    0.017629
feature_1113    0.016583
feature_1066    0.015568
feature_1180    0.014817
feature_1070    0.014656
feature_1166    0.013889
feature_1162    0.013701
feature_1124    0.013609
feature_1196    0.013436
feature_1136    0.012629
feature_1112    0.012055
feature_1109    0.011983
feature_1164    0.011586
feature_1134    0.011360
feature_1080    0.011061
feature_6       0.010874
Name: mutual_info, dtype: float64

## 7. LightGBM importance по CV

Это основной model-based рейтинг: он лучше корреляции ловит нелинейности и взаимодействия. Считаем importance по нескольким folds и усредняем.

In [10]:
def lgb_train_predict_importance(
    X: pd.DataFrame,
    y: np.ndarray,
    cv_splits: List[Tuple[np.ndarray, np.ndarray]],
    max_folds: int = 5,
    params: Optional[dict] = None,
) -> Tuple[pd.Series, pd.Series, np.ndarray, List[float]]:
    if not HAS_LIGHTGBM:
        return pd.Series(0.0, index=X.columns), pd.Series(0.0, index=X.columns), np.zeros(len(y)), []
    if params is None:
        pos = y.sum()
        neg = len(y) - pos
        params = {
            "objective": "binary",
            "metric": "auc",
            "learning_rate": 0.025,
            "num_leaves": 48,
            "min_data_in_leaf": 80,
            "feature_fraction": 0.80,
            "bagging_fraction": 0.85,
            "bagging_freq": 1,
            "lambda_l1": 0.1,
            "lambda_l2": 20.0,
            "scale_pos_weight": float(neg / max(pos, 1)),
            "verbosity": -1,
            "num_threads": N_JOBS,
            "seed": RANDOM_STATE,
            "force_col_wise": True,
        }

    gain = pd.Series(0.0, index=X.columns, dtype="float64")
    split = pd.Series(0.0, index=X.columns, dtype="float64")
    oof = np.zeros(len(y), dtype="float32")
    aucs: List[float] = []

    for fold, (tr_idx, va_idx) in enumerate(cv_splits[:max_folds]):
        print(f"LGB importance fold {fold}")
        dtr = lgb.Dataset(X.iloc[tr_idx], label=y[tr_idx], free_raw_data=False)
        dva = lgb.Dataset(X.iloc[va_idx], label=y[va_idx], reference=dtr, free_raw_data=False)
        model = lgb.train(
            params,
            dtr,
            num_boost_round=4000,
            valid_sets=[dva],
            valid_names=["valid"],
            callbacks=[lgb.early_stopping(120, verbose=False), lgb.log_evaluation(200)],
        )
        pred = model.predict(X.iloc[va_idx], num_iteration=model.best_iteration)
        oof[va_idx] = pred.astype("float32")
        auc = roc_auc_score(y[va_idx], pred)
        aucs.append(float(auc))
        print(f"fold={fold} auc={auc:.6f} best_iter={model.best_iteration}")

        gain += pd.Series(model.feature_importance("gain"), index=X.columns)
        split += pd.Series(model.feature_importance("split"), index=X.columns)
        del dtr, dva, model
        gc.collect()

    gain /= max(1, len(aucs))
    split /= max(1, len(aucs))
    gain.name = "lgb_gain"
    split.name = "lgb_split"
    return gain, split, oof, aucs

lgb_gain = pd.Series(0.0, index=X_train_raw_clean.columns, name="lgb_gain")
lgb_split = pd.Series(0.0, index=X_train_raw_clean.columns, name="lgb_split")
lgb_importance_oof = np.zeros(len(y), dtype="float32")
lgb_importance_aucs: List[float] = []

if RUN_LGB_IMPORTANCE:
    t0 = time.time()
    lgb_gain, lgb_split, lgb_importance_oof, lgb_importance_aucs = lgb_train_predict_importance(
        X_train_raw_clean,
        y,
        cv_splits,
        max_folds=LGB_IMPORTANCE_FOLDS,
    )
    print("LGB importance mean AUC:", np.mean(lgb_importance_aucs), "time", round(time.time() - t0, 1), "s")
    display(lgb_gain.sort_values(ascending=False).head(30))
else:
    print("RUN_LGB_IMPORTANCE=False")

LGB importance fold 0
[200]	valid's auc: 0.63427
fold=0 auc=0.635081 best_iter=185
LGB importance fold 1
[200]	valid's auc: 0.648285
fold=1 auc=0.649300 best_iter=225
LGB importance fold 2
[200]	valid's auc: 0.616044
[400]	valid's auc: 0.613332
fold=2 auc=0.617345 best_iter=284
LGB importance fold 3
[200]	valid's auc: 0.630408
fold=3 auc=0.631897 best_iter=240
LGB importance fold 4
[200]	valid's auc: 0.640741
fold=4 auc=0.641866 best_iter=226
LGB importance mean AUC: 0.6350977901734896 time 98.6 s


feature_41      59674.245200
feature_43      54816.089650
feature_1094    53470.004907
feature_13      37754.711906
feature_1122    37638.402264
feature_1091    37541.256056
feature_16      36586.393813
feature_22      35600.149823
feature_377     35060.498505
feature_1087    33191.804944
feature_11      32838.719171
feature_479     32074.907990
feature_1108    31868.304445
feature_1068    30959.753210
feature_37      30873.090974
feature_86      30269.048303
feature_46      30016.866597
feature_1078    29870.149188
feature_1075    29686.670914
feature_1076    29481.247597
feature_337     29220.326807
feature_42      28902.272379
feature_1085    28751.276221
feature_29      28747.176189
feature_44      28713.762300
feature_10      28703.475952
feature_1081    27774.795560
feature_1074    27257.227188
feature_34      26636.709607
feature_363     26546.743558
Name: lgb_gain, dtype: float64

## 8. Adversarial validation: какие признаки отличают train от test

Если признак сильно отличает train от test, он может ухудшать private generalization. Мы не удаляем такие признаки автоматически, но строим отдельные **stable feature sets**, где drifty-признаки исключены.

In [11]:
def adversarial_validation_importance(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    max_rows_each: int = 120_000,
) -> Tuple[pd.Series, float]:
    if not HAS_LIGHTGBM:
        return pd.Series(0.0, index=X_train.columns, name="adv_gain"), np.nan
    rng = np.random.default_rng(RANDOM_STATE)
    tr_idx = rng.choice(len(X_train), size=min(max_rows_each, len(X_train)), replace=False)
    te_idx = rng.choice(len(X_test), size=min(max_rows_each, len(X_test)), replace=False)
    X_adv = pd.concat([X_train.iloc[tr_idx], X_test.iloc[te_idx]], axis=0, ignore_index=True)
    y_adv = np.r_[np.zeros(len(tr_idx), dtype="int8"), np.ones(len(te_idx), dtype="int8")]

    tr_a, va_a = train_test_split(
        np.arange(len(y_adv)),
        test_size=0.25,
        random_state=RANDOM_STATE,
        stratify=y_adv,
    )
    params = {
        "objective": "binary",
        "metric": "auc",
        "learning_rate": 0.04,
        "num_leaves": 48,
        "min_data_in_leaf": 80,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "lambda_l2": 10,
        "verbosity": -1,
        "num_threads": N_JOBS,
        "seed": RANDOM_STATE,
        "force_col_wise": True,
    }
    dtr = lgb.Dataset(X_adv.iloc[tr_a], label=y_adv[tr_a], free_raw_data=False)
    dva = lgb.Dataset(X_adv.iloc[va_a], label=y_adv[va_a], reference=dtr, free_raw_data=False)
    model = lgb.train(
        params,
        dtr,
        num_boost_round=1200,
        valid_sets=[dva],
        valid_names=["valid"],
        callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(200)],
    )
    pred = model.predict(X_adv.iloc[va_a], num_iteration=model.best_iteration)
    auc = roc_auc_score(y_adv[va_a], pred)
    gain = pd.Series(model.feature_importance("gain"), index=X_train.columns, name="adv_gain")
    return gain, float(auc)

adv_gain = pd.Series(0.0, index=X_train_raw_clean.columns, name="adv_gain")
adv_auc = np.nan
if RUN_ADVERSARIAL_VALIDATION:
    t0 = time.time()
    adv_gain, adv_auc = adversarial_validation_importance(X_train_raw_clean, X_test_raw_clean)
    print("Adversarial validation AUC:", adv_auc, "time", round(time.time() - t0, 1), "s")
    display(adv_gain.sort_values(ascending=False).head(30))
else:
    print("RUN_ADVERSARIAL_VALIDATION=False")

Adversarial validation AUC: 0.5011296460787635 time 6.4 s


feature_37      60.820901
feature_1081    38.149900
feature_20      37.518100
feature_1072    36.321100
feature_824     35.266900
feature_1092    33.852300
feature_1065    31.749201
feature_805     30.781601
feature_1073    29.438300
feature_1114    28.985200
feature_484     28.617301
feature_1069    28.122600
feature_1193    27.829599
feature_1105    27.288700
feature_295     27.013801
feature_1161    26.041599
feature_10      25.881500
feature_1244    25.784600
feature_65      25.411901
feature_357     24.689100
feature_305     24.592700
feature_363     24.581599
feature_46      18.760201
feature_407     18.465799
feature_343     17.206100
feature_152     16.638700
feature_390     16.525801
feature_479     16.297899
feature_335     16.271200
feature_167     16.116699
Name: adv_gain, dtype: float64

## 9. Combined ranking и stable ranking

Собираем общий рейтинг из разных источников. Отдельно считаем `drift_score`, чтобы делать stable-наборы: высокая важность для target — хорошо, высокая важность для train-vs-test — риск.

In [12]:
def rank01(s: pd.Series, higher_is_better: bool = True) -> pd.Series:
    s = s.replace([np.inf, -np.inf], np.nan).fillna(0)
    if s.nunique() <= 1:
        return pd.Series(0.0, index=s.index)
    r = s.rank(ascending=not higher_is_better, method="average")
    return 1.0 - (r - 1) / max(len(r) - 1, 1)

feature_ranking = pd.DataFrame(index=X_train_raw_clean.columns)
feature_ranking["abs_corr"] = abs_corr.reindex(feature_ranking.index).fillna(0)
feature_ranking["single_feature_auc"] = single_auc.reindex(feature_ranking.index).fillna(0.5)
feature_ranking["single_auc_gain_over_random"] = (feature_ranking["single_feature_auc"] - 0.5).clip(lower=0)
feature_ranking["mutual_info"] = mi_scores.reindex(feature_ranking.index).fillna(0)
feature_ranking["lgb_gain"] = lgb_gain.reindex(feature_ranking.index).fillna(0)
feature_ranking["lgb_split"] = lgb_split.reindex(feature_ranking.index).fillna(0)
feature_ranking["adv_gain"] = adv_gain.reindex(feature_ranking.index).fillna(0)
feature_ranking = feature_ranking.join(feature_profile, how="left")

feature_ranking["rank_corr"] = rank01(feature_ranking["abs_corr"])
feature_ranking["rank_auc"] = rank01(feature_ranking["single_auc_gain_over_random"])
feature_ranking["rank_mi"] = rank01(feature_ranking["mutual_info"])
feature_ranking["rank_lgb_gain"] = rank01(np.log1p(feature_ranking["lgb_gain"]))
feature_ranking["rank_lgb_split"] = rank01(np.log1p(feature_ranking["lgb_split"]))
feature_ranking["rank_adv"] = rank01(np.log1p(feature_ranking["adv_gain"]))

feature_ranking["combined_score"] = (
    0.12 * feature_ranking["rank_corr"] +
    0.12 * feature_ranking["rank_auc"] +
    0.16 * feature_ranking["rank_mi"] +
    0.42 * feature_ranking["rank_lgb_gain"] +
    0.10 * feature_ranking["rank_lgb_split"] +
    0.08 * (1.0 - feature_ranking["rank_adv"])
)
feature_ranking["stable_score"] = feature_ranking["combined_score"] - 0.22 * feature_ranking["rank_adv"]
feature_ranking["drift_score"] = feature_ranking["rank_adv"]

feature_ranking = feature_ranking.sort_values("combined_score", ascending=False)
ranked_features = feature_ranking.index.tolist()
stable_ranked_features = feature_ranking.sort_values("stable_score", ascending=False).index.tolist()

drifty_features = feature_ranking.sort_values("drift_score", ascending=False).head(max(50, int(0.10 * len(feature_ranking)))).index.tolist()
non_drifty_ranked_features = [c for c in ranked_features if c not in set(drifty_features)]

feature_ranking.to_csv(OUTPUT_DIR / "feature_ranking_combined_v2.csv")
feature_profile.to_csv(OUTPUT_DIR / "feature_profile_full_v2.csv")
print("Saved ranking/profile to", OUTPUT_DIR)
print("Top combined features:")
display(feature_ranking.head(30))
print("Top stable features:")
display(feature_ranking.sort_values("stable_score", ascending=False).head(30))

Saved ranking/profile to /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/prepared_data
Top combined features:


,abs_corr,single_feature_auc,single_auc_gain_over_random,mutual_info,lgb_gain,lgb_split,adv_gain,zero_frac_train,zero_frac_test,zero_frac_delta,nunique_train,mean_train,mean_test,std_train,std_test,min_train,max_train,mean_abs_delta,std_ratio_train_test,rank_corr,rank_auc,rank_mi,rank_lgb_gain,rank_lgb_split,rank_adv,combined_score,stable_score,drift_score
feature_1087,0.015832,0.541136,0.041136,0.003492,33191.804944,73.0,0.0000,0.000000,0.000000,0.000000,201112,0.147538,0.147443,0.083700,0.083820,0.000573,9.269438e-01,0.000095,0.998578,0.995585,0.986019,0.951435,0.993377,0.984547,0.458793,0.948992,0.848057,0.458793
feature_1094,0.020181,0.550985,0.050985,0.002384,53470.004907,104.2,0.0000,0.000000,0.000000,0.000000,203027,0.267927,0.267716,0.169308,0.169274,0.000467,9.827985e-01,0.000211,1.000202,1.000000,0.999264,0.908756,0.998528,0.999264,0.458793,0.947918,0.846983,0.458793
feature_41,0.013425,0.532405,0.032405,0.002329,59674.245200,109.8,0.0000,0.000000,0.000000,0.000000,174004,0.765057,0.765291,0.133459,0.133765,0.116091,1.327086e+00,0.000234,0.997718,0.989698,0.963208,0.905077,1.000000,1.000000,0.458793,0.942458,0.841523,0.458793
feature_43,0.011912,0.527610,0.027610,0.002313,54816.089650,100.6,0.0000,0.000000,0.000000,0.000000,177664,0.502538,0.506474,0.754663,0.765221,0.007585,6.496452e+00,0.003936,0.986203,0.986019,0.932303,0.902870,0.999264,0.998528,0.458793,0.937498,0.836564,0.458793
feature_1076,0.007486,0.515806,0.015806,0.002700,29481.247597,87.2,0.0000,0.000000,0.000000,0.000000,186078,0.036515,0.036444,0.026772,0.026715,0.000124,6.624977e-01,0.000071,1.002140,0.933039,0.825607,0.930096,0.986019,0.993377,0.458793,0.916615,0.815681,0.458793
feature_35,0.008678,0.526710,0.026710,0.002419,19041.515002,46.4,0.0000,0.000000,0.000000,0.000000,4566,0.406602,0.407074,0.378571,0.379696,0.004602,2.727273e+00,0.000472,0.997037,0.958793,0.923473,0.910228,0.954378,0.958057,0.458793,0.911450,0.810515,0.458793
feature_377,0.011241,0.554352,0.054352,0.001042,35060.498505,38.4,0.0000,0.144004,0.141888,0.002116,8905,561.302979,586.779846,2643.070557,3474.119141,0.000000,2.129550e+05,25.476868,0.760789,0.982340,1.000000,0.735099,0.994113,0.944444,0.458793,0.910765,0.809831,0.458793
feature_1110,0.013611,0.537262,0.037262,0.005900,12347.292007,21.2,0.0000,0.000000,0.000000,0.000000,7,4.789210,4.788782,1.646821,1.648407,-1.000000,7.000000e+00,0.000428,0.999038,0.991906,0.979397,0.966152,0.921266,0.884106,0.458793,0.909779,0.808845,0.458793
feature_1101,0.007383,0.518606,0.018606,0.002509,21773.132755,65.8,0.0000,0.000000,0.000000,0.000000,189322,0.036459,0.036350,0.029171,0.029052,0.000214,7.941545e-01,0.000109,1.004105,0.931567,0.860927,0.916851,0.968359,0.974246,0.458793,0.909227,0.808293,0.458793
feature_1078,0.007646,0.516230,0.016230,0.001957,29870.149188,85.4,0.0000,0.000000,0.000000,0.000000,193678,0.030541,0.030735,0.041289,0.041513,0.000087,9.631388e-01,0.000193,0.994602,0.935247,0.832230,0.868286,0.987491,0.991906,0.458793,0.908256,0.807322,0.458793


Top stable features:


,abs_corr,single_feature_auc,single_auc_gain_over_random,mutual_info,lgb_gain,lgb_split,adv_gain,zero_frac_train,zero_frac_test,zero_frac_delta,nunique_train,mean_train,mean_test,std_train,std_test,min_train,max_train,mean_abs_delta,std_ratio_train_test,rank_corr,rank_auc,rank_mi,rank_lgb_gain,rank_lgb_split,rank_adv,combined_score,stable_score,drift_score
feature_1087,0.015832,0.541136,0.041136,0.003492,33191.804944,73.0,0.0,0.000000,0.000000,0.000000,201112,0.147538,0.147443,0.083700,0.083820,0.000573,9.269438e-01,0.000095,0.998578,0.995585,0.986019,0.951435,0.993377,0.984547,0.458793,0.948992,0.848057,0.458793
feature_1094,0.020181,0.550985,0.050985,0.002384,53470.004907,104.2,0.0,0.000000,0.000000,0.000000,203027,0.267927,0.267716,0.169308,0.169274,0.000467,9.827985e-01,0.000211,1.000202,1.000000,0.999264,0.908756,0.998528,0.999264,0.458793,0.947918,0.846983,0.458793
feature_41,0.013425,0.532405,0.032405,0.002329,59674.245200,109.8,0.0,0.000000,0.000000,0.000000,174004,0.765057,0.765291,0.133459,0.133765,0.116091,1.327086e+00,0.000234,0.997718,0.989698,0.963208,0.905077,1.000000,1.000000,0.458793,0.942458,0.841523,0.458793
feature_43,0.011912,0.527610,0.027610,0.002313,54816.089650,100.6,0.0,0.000000,0.000000,0.000000,177664,0.502538,0.506474,0.754663,0.765221,0.007585,6.496452e+00,0.003936,0.986203,0.986019,0.932303,0.902870,0.999264,0.998528,0.458793,0.937498,0.836564,0.458793
feature_1076,0.007486,0.515806,0.015806,0.002700,29481.247597,87.2,0.0,0.000000,0.000000,0.000000,186078,0.036515,0.036444,0.026772,0.026715,0.000124,6.624977e-01,0.000071,1.002140,0.933039,0.825607,0.930096,0.986019,0.993377,0.458793,0.916615,0.815681,0.458793
feature_35,0.008678,0.526710,0.026710,0.002419,19041.515002,46.4,0.0,0.000000,0.000000,0.000000,4566,0.406602,0.407074,0.378571,0.379696,0.004602,2.727273e+00,0.000472,0.997037,0.958793,0.923473,0.910228,0.954378,0.958057,0.458793,0.911450,0.810515,0.458793
feature_377,0.011241,0.554352,0.054352,0.001042,35060.498505,38.4,0.0,0.144004,0.141888,0.002116,8905,561.302979,586.779846,2643.070557,3474.119141,0.000000,2.129550e+05,25.476868,0.760789,0.982340,1.000000,0.735099,0.994113,0.944444,0.458793,0.910765,0.809831,0.458793
feature_1110,0.013611,0.537262,0.037262,0.005900,12347.292007,21.2,0.0,0.000000,0.000000,0.000000,7,4.789210,4.788782,1.646821,1.648407,-1.000000,7.000000e+00,0.000428,0.999038,0.991906,0.979397,0.966152,0.921266,0.884106,0.458793,0.909779,0.808845,0.458793
feature_1101,0.007383,0.518606,0.018606,0.002509,21773.132755,65.8,0.0,0.000000,0.000000,0.000000,189322,0.036459,0.036350,0.029171,0.029052,0.000214,7.941545e-01,0.000109,1.004105,0.931567,0.860927,0.916851,0.968359,0.974246,0.458793,0.909227,0.808293,0.458793
feature_1078,0.007646,0.516230,0.016230,0.001957,29870.149188,85.4,0.0,0.000000,0.000000,0.000000,193678,0.030541,0.030735,0.041289,0.041513,0.000087,9.631388e-01,0.000193,0.994602,0.935247,0.832230,0.868286,0.987491,0.991906,0.458793,0.908256,0.807322,0.458793


## 10. Row-level sparse/meta признаки

В таких данных часто важна не только конкретная колонка, но и «активность строки»: сколько ненулевых признаков, сколько положительных/отрицательных, как распределены значения, какая сумма/норма/максимум.

In [13]:
def add_row_meta_features(df: pd.DataFrame, prefix: str = "all") -> pd.DataFrame:
    arr = df.replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy(dtype=np.float32, copy=False)
    out = pd.DataFrame(index=df.index)
    zero = arr == 0
    pos = arr > 0
    neg = arr < 0
    abs_arr = np.abs(arr)
    logabs = np.log1p(abs_arr)

    n = arr.shape[1]
    out[f"{prefix}_zero_count"] = zero.sum(axis=1).astype("int16")
    out[f"{prefix}_nonzero_count"] = (n - out[f"{prefix}_zero_count"].values).astype("int16")
    out[f"{prefix}_zero_frac"] = (out[f"{prefix}_zero_count"].values / n).astype("float32")
    out[f"{prefix}_pos_count"] = pos.sum(axis=1).astype("int16")
    out[f"{prefix}_neg_count"] = neg.sum(axis=1).astype("int16")

    out[f"{prefix}_sum"] = arr.sum(axis=1).astype("float32")
    out[f"{prefix}_abs_sum"] = abs_arr.sum(axis=1).astype("float32")
    out[f"{prefix}_logabs_sum"] = logabs.sum(axis=1).astype("float32")
    out[f"{prefix}_mean"] = arr.mean(axis=1).astype("float32")
    out[f"{prefix}_std"] = arr.std(axis=1).astype("float32")
    out[f"{prefix}_abs_mean"] = abs_arr.mean(axis=1).astype("float32")
    out[f"{prefix}_abs_std"] = abs_arr.std(axis=1).astype("float32")
    out[f"{prefix}_max"] = arr.max(axis=1).astype("float32")
    out[f"{prefix}_min"] = arr.min(axis=1).astype("float32")
    out[f"{prefix}_abs_max"] = abs_arr.max(axis=1).astype("float32")
    out[f"{prefix}_l2_norm"] = np.sqrt((arr ** 2).sum(axis=1)).astype("float32")

    # Квантили считаются по абсолютным значениям, чтобы ловить "энергию" строки.
    for q in [0.50, 0.75, 0.90, 0.95, 0.99]:
        out[f"{prefix}_abs_q{int(q * 100):02d}"] = np.quantile(abs_arr, q, axis=1).astype("float32")

    # Энтропия распределения logabs по активным признакам: насколько строка сконцентрирована в нескольких фичах.
    denom = logabs.sum(axis=1, keepdims=True) + 1e-6
    p = logabs / denom
    entropy = -(p * np.log(p + 1e-9)).sum(axis=1)
    out[f"{prefix}_logabs_entropy"] = entropy.astype("float32")
    out[f"{prefix}_max_to_abs_sum"] = (out[f"{prefix}_abs_max"].values / (out[f"{prefix}_abs_sum"].values + 1e-6)).astype("float32")
    out[f"{prefix}_l2_to_l1"] = (out[f"{prefix}_l2_norm"].values / (out[f"{prefix}_abs_sum"].values + 1e-6)).astype("float32")

    return out

t0 = time.time()
train_meta = add_row_meta_features(X_train_raw_clean, "all")
test_meta = add_row_meta_features(X_test_raw_clean, "all")
print("Row meta:", train_meta.shape, test_meta.shape, "time", round(time.time() - t0, 1), "s")
display(train_meta.head())

Row meta: (247972, 24) (106274, 24) time 14.9 s


,all_zero_count,all_nonzero_count,all_zero_frac,all_pos_count,all_neg_count,all_sum,all_abs_sum,all_logabs_sum,all_mean,all_std,all_abs_mean,all_abs_std,all_max,all_min,all_abs_max,all_l2_norm,all_abs_q50,all_abs_q75,all_abs_q90,all_abs_q95,all_abs_q99,all_logabs_entropy,all_max_to_abs_sum,all_l2_to_l1
0,236,1124,0.173529,1104,20,95519.625000,95559.625000,1249.484741,70.235016,665.595337,70.264427,665.591675,13230.237305,-1.0,13230.237305,24682.248047,0.289128,1.810390,18.812820,74.731827,1804.004272,6.222245,0.138450,0.258292
1,78,1282,0.057353,1103,179,95358.242188,95716.242188,1358.515137,70.116356,665.609619,70.379593,665.577576,13230.237305,-1.0,13230.237305,24682.248047,0.646781,1.722785,18.812820,74.731827,1804.004272,6.408347,0.138224,0.257869
2,1096,264,0.805882,263,1,351827.031250,351829.031250,662.032593,258.696350,3116.980469,258.697815,3116.980225,53508.000000,-1.0,53508.000000,115344.515625,0.000000,0.000000,3.000000,39.049999,2669.879883,5.053093,0.152085,0.327843
3,1193,167,0.877206,161,6,54329.992188,54341.992188,279.391876,39.948524,499.825073,39.957348,499.824371,8144.000000,-1.0,8144.000000,18491.375000,0.000000,0.000000,0.084332,1.000000,371.100006,4.385954,0.149866,0.340278
4,1082,278,0.795588,277,1,437379.562500,437381.562500,581.026123,321.602631,4518.102051,321.604095,4518.102051,70242.000000,-1.0,70242.000000,167039.921875,0.000000,0.000000,1.094651,30.100000,807.159973,5.059719,0.160597,0.381909


## 11. Pattern frequency признаки

Если строки или zero/nonzero-паттерны повторяются, частота такого паттерна может быть сигналом. Эти признаки считаются по объединению train+test без target, поэтому target leakage нет.

In [14]:
def make_frequency_from_hash(train_hash: pd.Series, test_hash: pd.Series, name: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    all_hash = pd.concat([train_hash.astype("uint64"), test_hash.astype("uint64")], ignore_index=True)
    counts = all_hash.value_counts()
    train_out = pd.DataFrame(index=train_hash.index)
    test_out = pd.DataFrame(index=test_hash.index)
    train_out[f"{name}_freq"] = train_hash.map(counts).astype("float32").values
    test_out[f"{name}_freq"] = test_hash.map(counts).astype("float32").values
    train_out[f"{name}_logfreq"] = np.log1p(train_out[f"{name}_freq"]).astype("float32")
    test_out[f"{name}_logfreq"] = np.log1p(test_out[f"{name}_freq"]).astype("float32")
    return train_out, test_out

# full feature-vector frequency
train_fullhash_freq, test_fullhash_freq = make_frequency_from_hash(
    pd.Series(train_feature_hash.values, index=X_train_raw_clean.index),
    pd.Series(test_feature_hash.values, index=X_test_raw_clean.index),
    "full_vector_hash",
)

# zero/nonzero pattern hash. Более грубый, чем full hash.
# Чтобы не хранить огромную bool-таблицу, используем hash_pandas_object по boolean dataframe.
t0 = time.time()
train_zero_pattern_hash = hash_pandas_object(X_train_raw_clean.ne(0), index=False).astype("uint64")
test_zero_pattern_hash = hash_pandas_object(X_test_raw_clean.ne(0), index=False).astype("uint64")
train_zerohash_freq, test_zerohash_freq = make_frequency_from_hash(
    pd.Series(train_zero_pattern_hash.values, index=X_train_raw_clean.index),
    pd.Series(test_zero_pattern_hash.values, index=X_test_raw_clean.index),
    "zero_pattern_hash",
)
print("Pattern hash features time", round(time.time() - t0, 1), "s")

train_pattern_freq = pd.concat([train_fullhash_freq, train_zerohash_freq], axis=1)
test_pattern_freq = pd.concat([test_fullhash_freq, test_zerohash_freq], axis=1)
print("Pattern freq:", train_pattern_freq.shape)

Pattern hash features time 1.3 s
Pattern freq: (247972, 4)


## 12. Block statistics

Исходные `feature_0 ... feature_1366` могут быть не случайно упорядочены. Даже если нам неизвестен смысл блоков, статистики по блокам иногда ловят структуру источника данных.

Делаем два типа блоков:

- последовательные блоки по индексу фичи;
- блоки по топовым признакам из рейтинга.

In [15]:
def feature_number(col: str) -> int:
    try:
        return int(col.split("_")[-1])
    except Exception:
        return 10**9


def make_block_stats(df: pd.DataFrame, blocks: Dict[str, List[str]]) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    for name, cols in blocks.items():
        cols = [c for c in cols if c in df.columns]
        if not cols:
            continue
        arr = df[cols].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy(dtype=np.float32, copy=False)
        abs_arr = np.abs(arr)
        out[f"{name}_nonzero_count"] = (arr != 0).sum(axis=1).astype("int16")
        out[f"{name}_zero_frac"] = (arr == 0).mean(axis=1).astype("float32")
        out[f"{name}_sum"] = arr.sum(axis=1).astype("float32")
        out[f"{name}_abs_sum"] = abs_arr.sum(axis=1).astype("float32")
        out[f"{name}_mean"] = arr.mean(axis=1).astype("float32")
        out[f"{name}_std"] = arr.std(axis=1).astype("float32")
        out[f"{name}_max"] = arr.max(axis=1).astype("float32")
        out[f"{name}_abs_max"] = abs_arr.max(axis=1).astype("float32")
    return out

# последовательные блоки по номеру feature
sorted_raw_cols = sorted(X_train_raw_clean.columns, key=feature_number)
blocks: Dict[str, List[str]] = {}
for block_size in [50, 100, 200]:
    for start in range(0, len(sorted_raw_cols), block_size):
        cols = sorted_raw_cols[start:start + block_size]
        if len(cols) >= 10:
            blocks[f"seq_b{block_size}_{start:04d}_{start + len(cols) - 1:04d}"] = cols

# блоки по разным топ-k рейтингам
for k in [50, 100, 200, 500, 700, 1000]:
    blocks[f"ranked_top{k}"] = ranked_features[:min(k, len(ranked_features))]
    blocks[f"stable_top{k}"] = stable_ranked_features[:min(k, len(stable_ranked_features))]

print("Number of blocks:", len(blocks))
t0 = time.time()
train_block_meta = make_block_stats(X_train_raw_clean, blocks)
test_block_meta = make_block_stats(X_test_raw_clean, blocks)
print("Block meta:", train_block_meta.shape, test_block_meta.shape, "time", round(time.time() - t0, 1), "s")

Number of blocks: 61
Block meta: (247972, 488) (106274, 488) time 11.0 s


## 13. Top-k statistics по рейтингам

Берём несколько top-k наборов и считаем по ним агрегаты. Это даёт модели компактные признаки вида: «насколько строка активна именно в важных колонках».

In [16]:
def make_topk_stats(df: pd.DataFrame, feature_lists: Dict[str, List[str]]) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    for name, cols in feature_lists.items():
        cols = [c for c in cols if c in df.columns]
        if not cols:
            continue
        arr = df[cols].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy(dtype=np.float32, copy=False)
        abs_arr = np.abs(arr)
        logabs = np.log1p(abs_arr)
        out[f"{name}_nz"] = (arr != 0).sum(axis=1).astype("int16")
        out[f"{name}_pos"] = (arr > 0).sum(axis=1).astype("int16")
        out[f"{name}_neg"] = (arr < 0).sum(axis=1).astype("int16")
        out[f"{name}_sum"] = arr.sum(axis=1).astype("float32")
        out[f"{name}_abs_sum"] = abs_arr.sum(axis=1).astype("float32")
        out[f"{name}_logabs_sum"] = logabs.sum(axis=1).astype("float32")
        out[f"{name}_mean"] = arr.mean(axis=1).astype("float32")
        out[f"{name}_std"] = arr.std(axis=1).astype("float32")
        out[f"{name}_abs_max"] = abs_arr.max(axis=1).astype("float32")
        out[f"{name}_abs_q90"] = np.quantile(abs_arr, 0.90, axis=1).astype("float32")
        out[f"{name}_abs_q99"] = np.quantile(abs_arr, 0.99, axis=1).astype("float32")
    return out

topk_lists: Dict[str, List[str]] = {}
for k in [25, 50, 100, 200, 300, 500, 700, 1000]:
    topk_lists[f"combined_top{k}"] = ranked_features[:min(k, len(ranked_features))]
    topk_lists[f"stable_top{k}"] = stable_ranked_features[:min(k, len(stable_ranked_features))]
    topk_lists[f"nodrift_top{k}"] = non_drifty_ranked_features[:min(k, len(non_drifty_ranked_features))]

t0 = time.time()
train_topk_meta = make_topk_stats(X_train_raw_clean, topk_lists)
test_topk_meta = make_topk_stats(X_test_raw_clean, topk_lists)
print("Top-k meta:", train_topk_meta.shape, test_topk_meta.shape, "time", round(time.time() - t0, 1), "s")

Top-k meta: (247972, 264) (106274, 264) time 59.3 s


## 14. Frequency encoding отдельных признаков

Если признак имеет повторяющиеся значения, сама частота значения может быть полезна: редкое значение, популярное значение, повторяющийся паттерн и т.д. Считаем частоты на объединении train+test без target.

In [17]:
def make_frequency_encoding(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cols: List[str],
    round_decimals: int = 6,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_out = pd.DataFrame(index=train_df.index)
    test_out = pd.DataFrame(index=test_df.index)
    for col in cols:
        if col not in train_df.columns:
            continue
        tr = train_df[col].round(round_decimals)
        te = test_df[col].round(round_decimals)
        all_values = pd.concat([tr, te], ignore_index=True)
        freq = all_values.value_counts(dropna=False)
        name = col.replace("feature_", "f")
        train_out[f"freq_{name}"] = tr.map(freq).fillna(0).astype("float32").values
        test_out[f"freq_{name}"] = te.map(freq).fillna(0).astype("float32").values
        train_out[f"logfreq_{name}"] = np.log1p(train_out[f"freq_{name}"]).astype("float32")
        test_out[f"logfreq_{name}"] = np.log1p(test_out[f"freq_{name}"]).astype("float32")
    return train_out, test_out

freq_cols = [c for c in ranked_features if feature_ranking.loc[c, "nunique_train"] <= 5000][:FREQUENCY_ENCODING_TOP_N]
print("Frequency encoding cols:", len(freq_cols))

t0 = time.time()
train_freq, test_freq = make_frequency_encoding(X_train_raw_clean, X_test_raw_clean, freq_cols)
print("Frequency features:", train_freq.shape, test_freq.shape, "time", round(time.time() - t0, 1), "s")

Frequency encoding cols: 120
Frequency features: (247972, 240) (106274, 240) time 0.6 s


## 15. OOF target encoding

Target encoding опасен, если считать его на всём train: будет leakage. Здесь для train делаем строго OOF: каждая validation-часть кодируется статистикой только из train-части fold'а. Для test используем статистику по всему train.

Так как признаки числовые, перед TE округляем значения. Это превращает «точечные пики» и повторяющиеся значения в категориальные ключи.

In [18]:
def make_oof_target_encoding(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    y: np.ndarray,
    cols: List[str],
    cv_splits: List[Tuple[np.ndarray, np.ndarray]],
    round_decimals: int = 5,
    smoothing: float = 30.0,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    global_mean = float(np.mean(y))
    train_out = pd.DataFrame(index=train_df.index)
    test_out = pd.DataFrame(index=test_df.index)
    y_ser = pd.Series(y, index=train_df.index)

    for col in cols:
        if col not in train_df.columns:
            continue
        short = col.replace("feature_", "f")
        out_col = f"te_{short}"
        oof = pd.Series(global_mean, index=train_df.index, dtype="float32")
        tr_key_all = train_df[col].round(round_decimals)
        te_key = test_df[col].round(round_decimals)

        for tr_idx, va_idx in cv_splits:
            tr_keys = tr_key_all.iloc[tr_idx]
            va_keys = tr_key_all.iloc[va_idx]
            tr_y = y_ser.iloc[tr_idx]
            stats = pd.DataFrame({"key": tr_keys.values, "target": tr_y.values}).groupby("key")["target"].agg(["count", "mean"])
            smooth = (stats["count"] * stats["mean"] + smoothing * global_mean) / (stats["count"] + smoothing)
            oof.iloc[va_idx] = va_keys.map(smooth).fillna(global_mean).astype("float32").values

        full_stats = pd.DataFrame({"key": tr_key_all.values, "target": y}).groupby("key")["target"].agg(["count", "mean"])
        full_smooth = (full_stats["count"] * full_stats["mean"] + smoothing * global_mean) / (full_stats["count"] + smoothing)
        train_out[out_col] = oof.astype("float32")
        test_out[out_col] = te_key.map(full_smooth).fillna(global_mean).astype("float32").values
        train_out[f"{out_col}_delta_global"] = (train_out[out_col] - global_mean).astype("float32")
        test_out[f"{out_col}_delta_global"] = (test_out[out_col] - global_mean).astype("float32")

    return train_out, test_out

te_cols = [c for c in ranked_features if feature_ranking.loc[c, "nunique_train"] <= 5000][:TARGET_ENCODING_TOP_N]
print("Target encoding cols:", len(te_cols))

train_te = pd.DataFrame(index=X_train_raw_clean.index)
test_te = pd.DataFrame(index=X_test_raw_clean.index)
if RUN_TARGET_ENCODING and len(te_cols) > 0:
    t0 = time.time()
    train_te, test_te = make_oof_target_encoding(X_train_raw_clean, X_test_raw_clean, y, te_cols, cv_splits)
    print("TE features:", train_te.shape, test_te.shape, "time", round(time.time() - t0, 1), "s")
else:
    print("Target encoding skipped")

Target encoding cols: 80
TE features: (247972, 160) (106274, 160) time 3.2 s


## 16. Peak flags: автоматический поиск сильных повторяющихся значений

Некоторые признаки могут иметь конкретные значения, где доля positive сильно выше обычной. Для таких значений делаем бинарные флаги: `feature_x == suspicious_peak_value`.

In [19]:
def discover_peak_values(
    train_df: pd.DataFrame,
    y: np.ndarray,
    cols: List[str],
    round_decimals: int = 5,
    min_count: int = 25,
    min_positive_count: int = 3,
    min_lift: float = 3.0,
    max_values_per_feature: int = 4,
) -> pd.DataFrame:
    global_rate = float(np.mean(y))
    rows = []
    y_ser = pd.Series(y, index=train_df.index)
    for col in cols:
        keys = train_df[col].round(round_decimals)
        stats = pd.DataFrame({"key": keys.values, "target": y_ser.values}).groupby("key")["target"].agg(["count", "sum", "mean"])
        stats = stats[(stats["count"] >= min_count) & (stats["sum"] >= min_positive_count)].copy()
        if stats.empty:
            continue
        stats["lift"] = stats["mean"] / max(global_rate, 1e-12)
        stats = stats[stats["lift"] >= min_lift].sort_values(["lift", "sum", "count"], ascending=False).head(max_values_per_feature)
        for value, r in stats.iterrows():
            rows.append({
                "feature": col,
                "rounded_value": float(value),
                "count": int(r["count"]),
                "positive_count": int(r["sum"]),
                "target_mean": float(r["mean"]),
                "lift": float(r["lift"]),
            })
    return pd.DataFrame(rows).sort_values(["lift", "positive_count", "count"], ascending=False) if rows else pd.DataFrame(columns=["feature", "rounded_value", "count", "positive_count", "target_mean", "lift"])


def make_peak_flags(train_df: pd.DataFrame, test_df: pd.DataFrame, peaks: pd.DataFrame, round_decimals: int = 5) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_out = pd.DataFrame(index=train_df.index)
    test_out = pd.DataFrame(index=test_df.index)
    for i, row in peaks.reset_index(drop=True).iterrows():
        col = row["feature"]
        val = row["rounded_value"]
        if col not in train_df.columns:
            continue
        short = col.replace("feature_", "f")
        name = f"peak_{short}_{i:03d}"
        train_out[name] = (train_df[col].round(round_decimals) == round(val, round_decimals)).astype("int8").values
        test_out[name] = (test_df[col].round(round_decimals) == round(val, round_decimals)).astype("int8").values
    if train_out.shape[1] > 0:
        train_out["peak_flags_sum"] = train_out.sum(axis=1).astype("int16")
        test_out["peak_flags_sum"] = test_out.sum(axis=1).astype("int16")
    return train_out, test_out

peak_search_cols = ranked_features[:min(700, len(ranked_features))]
t0 = time.time()
peaks = discover_peak_values(X_train_raw_clean, y, peak_search_cols)
peaks.to_csv(OUTPUT_DIR / "discovered_peak_values_v2.csv", index=False)
print("Discovered peaks:", len(peaks), "time", round(time.time() - t0, 1), "s")
display(peaks.head(50))

train_peak_flags, test_peak_flags = make_peak_flags(X_train_raw_clean, X_test_raw_clean, peaks.head(300))
print("Peak flags:", train_peak_flags.shape, test_peak_flags.shape)

Discovered peaks: 1074 time 6.0 s


,feature,rounded_value,count,positive_count,target_mean,lift
665,feature_262,222.00000,26,5,0.192308,14.251920
180,feature_1081,0.01827,27,5,0.185185,13.724071
811,feature_1073,0.03783,27,5,0.185185,13.724071
35,feature_13,0.56072,29,5,0.172414,12.777583
261,feature_20,0.36662,41,7,0.170732,12.652924
255,feature_1070,0.00674,30,5,0.166667,12.351664
87,feature_1068,0.07930,25,4,0.160000,11.857597
130,feature_195,1814.00000,25,4,0.160000,11.857597
36,feature_13,0.61621,26,4,0.153846,11.401536
67,feature_39,0.25175,26,4,0.153846,11.401536


Peak flags: (247972, 301) (106274, 301)


## 17. Pairwise interactions для топовых признаков

Полный перебор пар невозможен. Берём только топовые признаки по combined ranking и создаём несколько простых взаимодействий: сумма, разность, произведение signed-log значений, оба ненулевые, отношение лог-масштабов.

In [20]:
def make_pairwise_interactions(df: pd.DataFrame, cols: List[str], prefix: str = "int") -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    cols = [c for c in cols if c in df.columns]
    data = df[cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    signed_log = np.sign(data.to_numpy(dtype=np.float32, copy=False)) * np.log1p(np.abs(data.to_numpy(dtype=np.float32, copy=False)))
    raw = data.to_numpy(dtype=np.float32, copy=False)
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            a_col, b_col = cols[i], cols[j]
            a = raw[:, i]
            b = raw[:, j]
            la = signed_log[:, i]
            lb = signed_log[:, j]
            short_a = a_col.replace("feature_", "f")
            short_b = b_col.replace("feature_", "f")
            base = f"{prefix}_{short_a}_{short_b}"
            out[f"{base}_sum"] = (a + b).astype("float32")
            out[f"{base}_diff"] = (a - b).astype("float32")
            out[f"{base}_logdiff"] = (la - lb).astype("float32")
            out[f"{base}_logprod"] = (la * lb).astype("float32")
            out[f"{base}_both_nz"] = ((a != 0) & (b != 0)).astype("int8")
    return out

train_interactions = pd.DataFrame(index=X_train_raw_clean.index)
test_interactions = pd.DataFrame(index=X_test_raw_clean.index)
if RUN_INTERACTIONS:
    inter_cols = ranked_features[:min(INTERACTION_TOP_N, len(ranked_features))]
    print("Interaction cols:", len(inter_cols))
    t0 = time.time()
    train_interactions = make_pairwise_interactions(X_train_raw_clean, inter_cols, "pair")
    test_interactions = make_pairwise_interactions(X_test_raw_clean, inter_cols, "pair")
    print("Interactions:", train_interactions.shape, test_interactions.shape, "time", round(time.time() - t0, 1), "s")
else:
    print("RUN_INTERACTIONS=False")

Interaction cols: 32
Interactions: (247972, 2480) (106274, 2480) time 3.4 s


## 18. PCA / Quantile-PCA / SVD

Делаем несколько проекций:

1. `PCA` по signed-log значениям топовых признаков.
2. `QuantileTransformer + PCA`: полезнее для линейных моделей, потому что распределения становятся более нормальными.
3. `TruncatedSVD` по binary nonzero mask: отдельный взгляд на паттерн активности, а не на величины.

In [21]:
def signed_log1p_frame(df: pd.DataFrame) -> np.ndarray:
    arr = df.replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy(dtype=np.float32, copy=False)
    return np.sign(arr) * np.log1p(np.abs(arr))


def make_pca_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cols: List[str],
    n_components: int,
    prefix: str,
    use_quantile: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    cols = [c for c in cols if c in train_df.columns]
    n_components = min(n_components, len(cols), len(train_df) - 1)
    if n_components <= 0:
        return pd.DataFrame(index=train_df.index), pd.DataFrame(index=test_df.index)
    Xtr = signed_log1p_frame(train_df[cols])
    Xte = signed_log1p_frame(test_df[cols])

    if use_quantile:
        qt = QuantileTransformer(
            n_quantiles=min(1000, len(train_df)),
            output_distribution="normal",
            random_state=RANDOM_STATE,
            subsample=min(200_000, len(train_df)),
        )
        Xtr = qt.fit_transform(Xtr).astype("float32")
        Xte = qt.transform(Xte).astype("float32")
    else:
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(Xtr).astype("float32")
        Xte = scaler.transform(Xte).astype("float32")

    pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
    tr_comp = pca.fit_transform(Xtr).astype("float32")
    te_comp = pca.transform(Xte).astype("float32")
    names = [f"{prefix}_{i:03d}" for i in range(n_components)]
    train_out = pd.DataFrame(tr_comp, columns=names, index=train_df.index)
    test_out = pd.DataFrame(te_comp, columns=names, index=test_df.index)
    evr = pd.DataFrame({"component": names, "explained_variance_ratio": pca.explained_variance_ratio_})
    evr.to_csv(OUTPUT_DIR / f"{prefix}_explained_variance.csv", index=False)
    print(prefix, "n_components", n_components, "explained sum", float(pca.explained_variance_ratio_.sum()))
    return train_out, test_out

train_pca = pd.DataFrame(index=X_train_raw_clean.index)
test_pca = pd.DataFrame(index=X_test_raw_clean.index)
train_qpca = pd.DataFrame(index=X_train_raw_clean.index)
test_qpca = pd.DataFrame(index=X_test_raw_clean.index)
if RUN_PCA:
    t0 = time.time()
    train_pca, test_pca = make_pca_features(
        X_train_raw_clean, X_test_raw_clean,
        ranked_features[:min(PCA_TOP_N, len(ranked_features))],
        PCA_N_COMPONENTS,
        "pca_log_top",
        use_quantile=False,
    )
    train_qpca, test_qpca = make_pca_features(
        X_train_raw_clean, X_test_raw_clean,
        ranked_features[:min(QUANTILE_PCA_TOP_N, len(ranked_features))],
        QUANTILE_PCA_N_COMPONENTS,
        "qpca_top",
        use_quantile=True,
    )
    print("PCA total time", round(time.time() - t0, 1), "s")
else:
    print("RUN_PCA=False")

pca_log_top n_components 48 explained sum 0.616415798664093
qpca_top n_components 32 explained sum 0.6749825477600098
PCA total time 15.4 s


In [22]:
def make_svd_nonzero_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cols: List[str],
    n_components: int,
    prefix: str = "svd_nz",
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if not HAS_SCIPY:
        return pd.DataFrame(index=train_df.index), pd.DataFrame(index=test_df.index)
    cols = [c for c in cols if c in train_df.columns]
    n_components = min(n_components, len(cols) - 1)
    if n_components <= 0:
        return pd.DataFrame(index=train_df.index), pd.DataFrame(index=test_df.index)
    tr_mask = sp.csr_matrix(train_df[cols].ne(0).astype("float32").values)
    te_mask = sp.csr_matrix(test_df[cols].ne(0).astype("float32").values)
    svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
    tr_comp = svd.fit_transform(tr_mask).astype("float32")
    te_comp = svd.transform(te_mask).astype("float32")
    names = [f"{prefix}_{i:03d}" for i in range(n_components)]
    train_out = pd.DataFrame(tr_comp, columns=names, index=train_df.index)
    test_out = pd.DataFrame(te_comp, columns=names, index=test_df.index)
    evr = pd.DataFrame({"component": names, "explained_variance_ratio": svd.explained_variance_ratio_})
    evr.to_csv(OUTPUT_DIR / f"{prefix}_explained_variance.csv", index=False)
    print(prefix, "n_components", n_components, "explained sum", float(svd.explained_variance_ratio_.sum()))
    return train_out, test_out

train_svd = pd.DataFrame(index=X_train_raw_clean.index)
test_svd = pd.DataFrame(index=X_test_raw_clean.index)
if RUN_SVD:
    t0 = time.time()
    svd_cols = ranked_features[:min(SVD_TOP_N, len(ranked_features))]
    train_svd, test_svd = make_svd_nonzero_features(X_train_raw_clean, X_test_raw_clean, svd_cols, SVD_N_COMPONENTS, "svd_nz_top")
    print("SVD time", round(time.time() - t0, 1), "s")
else:
    print("RUN_SVD=False")

svd_nz_top n_components 64 explained sum 0.7932739853858948
SVD time 12.3 s


## 19. KMeans-сегменты в PCA-пространстве

Кластеризация даёт модели грубый «тип строки». Используем PCA/Quantile-PCA компоненты, а не исходные 1000+ фичей.

In [23]:
def make_kmeans_features(
    train_comp: pd.DataFrame,
    test_comp: pd.DataFrame,
    n_clusters: int,
    prefix: str,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if train_comp.shape[1] == 0:
        return pd.DataFrame(index=train_comp.index), pd.DataFrame(index=test_comp.index)
    n_clusters = min(n_clusters, max(2, len(train_comp) // 1000))
    km = MiniBatchKMeans(
        n_clusters=n_clusters,
        random_state=RANDOM_STATE,
        batch_size=8192,
        n_init="auto",
    )
    km.fit(train_comp.values)
    tr_dist = km.transform(train_comp.values).astype("float32")
    te_dist = km.transform(test_comp.values).astype("float32")
    train_out = pd.DataFrame(index=train_comp.index)
    test_out = pd.DataFrame(index=test_comp.index)
    train_out[f"{prefix}_cluster"] = tr_dist.argmin(axis=1).astype("int16")
    test_out[f"{prefix}_cluster"] = te_dist.argmin(axis=1).astype("int16")
    train_out[f"{prefix}_min_dist"] = tr_dist.min(axis=1).astype("float32")
    test_out[f"{prefix}_min_dist"] = te_dist.min(axis=1).astype("float32")
    # Расстояния до первых нескольких центров — как мягкие признаки сегментов.
    keep = min(8, n_clusters)
    for i in range(keep):
        train_out[f"{prefix}_dist_{i:02d}"] = tr_dist[:, i].astype("float32")
        test_out[f"{prefix}_dist_{i:02d}"] = te_dist[:, i].astype("float32")
    return train_out, test_out

train_kmeans = pd.DataFrame(index=X_train_raw_clean.index)
test_kmeans = pd.DataFrame(index=X_test_raw_clean.index)
if RUN_KMEANS:
    base_comp = pd.concat([train_pca, train_qpca], axis=1)
    base_test_comp = pd.concat([test_pca, test_qpca], axis=1)
    t0 = time.time()
    train_kmeans, test_kmeans = make_kmeans_features(base_comp, base_test_comp, KMEANS_CLUSTERS, "km_pca")
    print("KMeans features:", train_kmeans.shape, test_kmeans.shape, "time", round(time.time() - t0, 1), "s")
else:
    print("RUN_KMEANS=False")

KMeans features: (247972, 10) (106274, 10) time 0.4 s


## 20. Model-based OOF meta-features

Это уже почти stacking-признаки. Для train используем OOF-предсказания, для test — модель, обученную на всём train. Это честно: train-строка получает предсказание модели, которая не видела её target.

Такие признаки особенно полезны для финального stacking/blending.

In [24]:
def fit_lgb_oof_and_test(
    X: pd.DataFrame,
    X_test: pd.DataFrame,
    y: np.ndarray,
    cv_splits: List[Tuple[np.ndarray, np.ndarray]],
    params: Optional[dict] = None,
    name: str = "rough_lgb",
) -> Tuple[np.ndarray, np.ndarray, List[float]]:
    if not HAS_LIGHTGBM:
        return np.zeros(len(y), dtype="float32"), np.zeros(len(X_test), dtype="float32"), []
    pos = y.sum(); neg = len(y) - pos
    if params is None:
        params = {
            "objective": "binary",
            "metric": "auc",
            "learning_rate": 0.018,
            "num_leaves": 40,
            "min_data_in_leaf": 100,
            "feature_fraction": 0.75,
            "bagging_fraction": 0.85,
            "bagging_freq": 1,
            "lambda_l1": 0.2,
            "lambda_l2": 30,
            "scale_pos_weight": float(neg / max(pos, 1)),
            "verbosity": -1,
            "num_threads": N_JOBS,
            "seed": RANDOM_STATE + 777,
            "force_col_wise": True,
        }
    oof = np.zeros(len(y), dtype="float32")
    test_preds = []
    aucs = []
    for fold, (tr_idx, va_idx) in enumerate(cv_splits):
        print(f"{name} fold {fold}")
        dtr = lgb.Dataset(X.iloc[tr_idx], label=y[tr_idx], free_raw_data=False)
        dva = lgb.Dataset(X.iloc[va_idx], label=y[va_idx], reference=dtr, free_raw_data=False)
        model = lgb.train(
            params,
            dtr,
            num_boost_round=5000,
            valid_sets=[dva],
            valid_names=["valid"],
            callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(300)],
        )
        pred_va = model.predict(X.iloc[va_idx], num_iteration=model.best_iteration)
        pred_te = model.predict(X_test, num_iteration=model.best_iteration)
        oof[va_idx] = pred_va.astype("float32")
        test_preds.append(pred_te.astype("float32"))
        auc = roc_auc_score(y[va_idx], pred_va)
        aucs.append(float(auc))
        print(f"{name} fold={fold} auc={auc:.6f} best_iter={model.best_iteration}")
        del dtr, dva, model
        gc.collect()
    return oof, np.mean(test_preds, axis=0).astype("float32"), aucs


def fit_logreg_oof_and_test(
    X: pd.DataFrame,
    X_test: pd.DataFrame,
    y: np.ndarray,
    cv_splits: List[Tuple[np.ndarray, np.ndarray]],
    name: str = "rough_lr",
) -> Tuple[np.ndarray, np.ndarray, List[float]]:
    oof = np.zeros(len(y), dtype="float32")
    test_preds = []
    aucs = []
    for fold, (tr_idx, va_idx) in enumerate(cv_splits):
        print(f"{name} fold {fold}")
        model = make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            LogisticRegression(
                C=0.5,
                penalty="l2",
                solver="saga",
                max_iter=1000,
                class_weight="balanced",
                n_jobs=N_JOBS,
                random_state=RANDOM_STATE + fold,
            ),
        )
        model.fit(X.iloc[tr_idx], y[tr_idx])
        pred_va = model.predict_proba(X.iloc[va_idx])[:, 1]
        pred_te = model.predict_proba(X_test)[:, 1]
        oof[va_idx] = pred_va.astype("float32")
        test_preds.append(pred_te.astype("float32"))
        auc = roc_auc_score(y[va_idx], pred_va)
        aucs.append(float(auc))
        print(f"{name} fold={fold} auc={auc:.6f}")
        del model
        gc.collect()
    return oof, np.mean(test_preds, axis=0).astype("float32"), aucs

train_model_meta = pd.DataFrame(index=X_train_raw_clean.index)
test_model_meta = pd.DataFrame(index=X_test_raw_clean.index)
model_meta_report = []

if RUN_MODEL_META_FEATURES:
    # Для meta-model признаков используем не слишком широкий, но сильный набор.
    meta_base_cols = ranked_features[:min(700, len(ranked_features))]
    X_meta_base = pd.concat([
        X_train_raw_clean[meta_base_cols],
        train_meta,
        train_topk_meta,
        train_peak_flags,
        train_freq,
    ], axis=1).loc[:, lambda d: ~d.columns.duplicated()].copy()
    X_test_meta_base = pd.concat([
        X_test_raw_clean[meta_base_cols],
        test_meta,
        test_topk_meta,
        test_peak_flags,
        test_freq,
    ], axis=1).reindex(columns=X_meta_base.columns, fill_value=0)

    t0 = time.time()
    if HAS_LIGHTGBM:
        oof_lgb, test_lgb, aucs_lgb = fit_lgb_oof_and_test(X_meta_base, X_test_meta_base, y, cv_splits, name="meta_lgb")
        train_model_meta["meta_lgb_oof"] = oof_lgb
        test_model_meta["meta_lgb_oof"] = test_lgb
        train_model_meta["meta_lgb_logit"] = np.log(np.clip(oof_lgb, 1e-6, 1 - 1e-6) / np.clip(1 - oof_lgb, 1e-6, 1)).astype("float32")
        test_model_meta["meta_lgb_logit"] = np.log(np.clip(test_lgb, 1e-6, 1 - 1e-6) / np.clip(1 - test_lgb, 1e-6, 1)).astype("float32")
        model_meta_report.append({"model": "meta_lgb", "mean_auc": float(np.mean(aucs_lgb)), "std_auc": float(np.std(aucs_lgb))})

    # Логрег на compact/PCA/TE признаках
    lr_base_parts = [
        train_meta,
        train_topk_meta,
        train_pca,
        train_qpca,
        train_svd,
        train_te,
    ]
    lr_test_parts = [
        test_meta,
        test_topk_meta,
        test_pca,
        test_qpca,
        test_svd,
        test_te,
    ]
    X_lr_base = pd.concat(lr_base_parts, axis=1).loc[:, lambda d: ~d.columns.duplicated()].copy()
    X_test_lr_base = pd.concat(lr_test_parts, axis=1).reindex(columns=X_lr_base.columns, fill_value=0)
    if X_lr_base.shape[1] > 0:
        oof_lr, test_lr, aucs_lr = fit_logreg_oof_and_test(X_lr_base, X_test_lr_base, y, cv_splits, name="meta_lr")
        train_model_meta["meta_lr_oof"] = oof_lr
        test_model_meta["meta_lr_oof"] = test_lr
        train_model_meta["meta_lr_logit"] = np.log(np.clip(oof_lr, 1e-6, 1 - 1e-6) / np.clip(1 - oof_lr, 1e-6, 1)).astype("float32")
        test_model_meta["meta_lr_logit"] = np.log(np.clip(test_lr, 1e-6, 1 - 1e-6) / np.clip(1 - test_lr, 1e-6, 1)).astype("float32")
        model_meta_report.append({"model": "meta_lr", "mean_auc": float(np.mean(aucs_lr)), "std_auc": float(np.std(aucs_lr))})

    pd.DataFrame(model_meta_report).to_csv(OUTPUT_DIR / "model_meta_report.csv", index=False)
    print("Model meta features:", train_model_meta.shape, test_model_meta.shape, "time", round(time.time() - t0, 1), "s")
    display(pd.DataFrame(model_meta_report))
else:
    print("RUN_MODEL_META_FEATURES=False")

meta_lgb fold 0
[300]	valid's auc: 0.700131
meta_lgb fold=0 auc=0.700680 best_iter=322
meta_lgb fold 1
[300]	valid's auc: 0.711659
[600]	valid's auc: 0.713043
meta_lgb fold=1 auc=0.713617 best_iter=496
meta_lgb fold 2
[300]	valid's auc: 0.700322
[600]	valid's auc: 0.702475
meta_lgb fold=2 auc=0.703249 best_iter=627
meta_lgb fold 3
[300]	valid's auc: 0.710161
meta_lgb fold=3 auc=0.710297 best_iter=299
meta_lgb fold 4
[300]	valid's auc: 0.694089
meta_lgb fold=4 auc=0.694690 best_iter=385
meta_lr fold 0
meta_lr fold=0 auc=0.655379
meta_lr fold 1
meta_lr fold=1 auc=0.642232
meta_lr fold 2
meta_lr fold=2 auc=0.628925
meta_lr fold 3
meta_lr fold=3 auc=0.651539
meta_lr fold 4
meta_lr fold=4 auc=0.627107
Model meta features: (247972, 4) (106274, 4) time 776.1 s


,model,mean_auc,std_auc
0,meta_lgb,0.704507,0.006769
1,meta_lr,0.641036,0.011473


## 21. Noise candidates

Если OOF-модель уверенно спорит с target, строка может быть сложной или шумной. Мы ничего не удаляем прямо здесь, но сохраняем таблицу кандидатов — дальше можно пробовать `sample_weight` или отдельные clean-feature experiments.

In [25]:
noise_df = pd.DataFrame({ID_COL: train[ID_COL].values, TARGET_COL: y})
if "meta_lgb_oof" in train_model_meta.columns:
    p = train_model_meta["meta_lgb_oof"].values
elif len(lgb_importance_aucs) > 0:
    p = lgb_importance_oof
else:
    p = np.full(len(y), y.mean(), dtype="float32")

noise_df["oof_pred"] = p
noise_df["suspicious_score"] = np.where(y == 1, 1 - p, p).astype("float32")
noise_df["is_top_001_noise"] = (noise_df["suspicious_score"] >= noise_df["suspicious_score"].quantile(0.999)).astype("int8")
noise_df["is_top_005_noise"] = (noise_df["suspicious_score"] >= noise_df["suspicious_score"].quantile(0.995)).astype("int8")
noise_df["is_top_010_noise"] = (noise_df["suspicious_score"] >= noise_df["suspicious_score"].quantile(0.990)).astype("int8")
noise_df = noise_df.sort_values("suspicious_score", ascending=False)
noise_df.to_csv(OUTPUT_DIR / "noise_candidates_by_oof.csv", index=False)
print("Saved:", OUTPUT_DIR / "noise_candidates_by_oof.csv")
display(noise_df.head(30))

Saved: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/prepared_data/noise_candidates_by_oof.csv


,index,target,oof_pred,suspicious_score,is_top_001_noise,is_top_005_noise,is_top_010_noise
113881,71146,1,0.014456,0.985544,1,1,1
201487,350068,1,0.016145,0.983855,1,1,1
167097,237058,1,0.018290,0.981710,1,1,1
106573,148648,1,0.020280,0.979720,1,1,1
239353,150834,1,0.020571,0.979429,1,1,1
171943,234604,1,0.021594,0.978406,1,1,1
55416,49807,1,0.021890,0.978110,1,1,1
143262,104919,1,0.023481,0.976519,1,1,1
142889,140120,1,0.023976,0.976025,1,1,1
15257,161822,1,0.025092,0.974908,1,1,1


## 22. Сборка engineered matrix

Склеиваем все новые признаки. Все признаки синхронизируются между train/test, бесконечности заменяются на `NaN`, затем сохраняются. Импьютацию лучше делать внутри модельного пайплайна, но для большинства engineered-признаков NaN возникать не должен.

In [26]:
engineered_parts_train = [
    train_meta,
    train_pattern_freq,
    train_block_meta,
    train_topk_meta,
    train_freq,
    train_te,
    train_peak_flags,
    train_interactions,
    train_pca,
    train_qpca,
    train_svd,
    train_kmeans,
    train_model_meta,
]
engineered_parts_test = [
    test_meta,
    test_pattern_freq,
    test_block_meta,
    test_topk_meta,
    test_freq,
    test_te,
    test_peak_flags,
    test_interactions,
    test_pca,
    test_qpca,
    test_svd,
    test_kmeans,
    test_model_meta,
]

X_train_engineered = pd.concat(engineered_parts_train, axis=1)
X_test_engineered = pd.concat(engineered_parts_test, axis=1)

X_train_engineered = X_train_engineered.loc[:, ~X_train_engineered.columns.duplicated()].copy()
X_test_engineered = X_test_engineered.loc[:, ~X_test_engineered.columns.duplicated()].copy()
X_test_engineered = X_test_engineered.reindex(columns=X_train_engineered.columns, fill_value=0)

# Базовая санитизация типов
for df in [X_train_engineered, X_test_engineered]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    for c in df.select_dtypes(include=["float64"]).columns:
        df[c] = df[c].astype("float32")

print("X_train_engineered:", X_train_engineered.shape)
print("X_test_engineered:", X_test_engineered.shape)
print("Missing engineered train:", int(X_train_engineered.isna().sum().sum()))
print("Missing engineered test:", int(X_test_engineered.isna().sum().sum()))
display(X_train_engineered.head())

X_train_engineered: (247972, 4089)
X_test_engineered: (106274, 4089)
Missing engineered train: 0
Missing engineered test: 0


,all_zero_count,all_nonzero_count,all_zero_frac,all_pos_count,all_neg_count,all_sum,all_abs_sum,all_logabs_sum,all_mean,all_std,all_abs_mean,all_abs_std,all_max,all_min,all_abs_max,all_l2_norm,all_abs_q50,all_abs_q75,all_abs_q90,all_abs_q95,all_abs_q99,all_logabs_entropy,all_max_to_abs_sum,all_l2_to_l1,full_vector_hash_freq,full_vector_hash_logfreq,zero_pattern_hash_freq,zero_pattern_hash_logfreq,seq_b50_0000_0049_nonzero_count,seq_b50_0000_0049_zero_frac,seq_b50_0000_0049_sum,seq_b50_0000_0049_abs_sum,seq_b50_0000_0049_mean,seq_b50_0000_0049_std,seq_b50_0000_0049_max,seq_b50_0000_0049_abs_max,seq_b50_0050_0099_nonzero_count,seq_b50_0050_0099_zero_frac,seq_b50_0050_0099_sum,seq_b50_0050_0099_abs_sum,seq_b50_0050_0099_mean,seq_b50_0050_0099_std,seq_b50_0050_0099_max,seq_b50_0050_0099_abs_max,seq_b50_0100_0149_nonzero_count,seq_b50_0100_0149_zero_frac,seq_b50_0100_0149_sum,seq_b50_0100_0149_abs_sum,seq_b50_0100_0149_mean,seq_b50_0100_0149_std,seq_b50_0100_0149_max,seq_b50_0100_0149_abs_max,seq_b50_0150_0199_nonzero_count,seq_b50_0150_0199_zero_frac,seq_b50_0150_0199_sum,seq_b50_0150_0199_abs_sum,seq_b50_0150_0199_mean,seq_b50_0150_0199_std,seq_b50_0150_0199_max,seq_b50_0150_0199_abs_max,...,svd_nz_top_018,svd_nz_top_019,svd_nz_top_020,svd_nz_top_021,svd_nz_top_022,svd_nz_top_023,svd_nz_top_024,svd_nz_top_025,svd_nz_top_026,svd_nz_top_027,svd_nz_top_028,svd_nz_top_029,svd_nz_top_030,svd_nz_top_031,svd_nz_top_032,svd_nz_top_033,svd_nz_top_034,svd_nz_top_035,svd_nz_top_036,svd_nz_top_037,svd_nz_top_038,svd_nz_top_039,svd_nz_top_040,svd_nz_top_041,svd_nz_top_042,svd_nz_top_043,svd_nz_top_044,svd_nz_top_045,svd_nz_top_046,svd_nz_top_047,svd_nz_top_048,svd_nz_top_049,svd_nz_top_050,svd_nz_top_051,svd_nz_top_052,svd_nz_top_053,svd_nz_top_054,svd_nz_top_055,svd_nz_top_056,svd_nz_top_057,svd_nz_top_058,svd_nz_top_059,svd_nz_top_060,svd_nz_top_061,svd_nz_top_062,svd_nz_top_063,km_pca_cluster,km_pca_min_dist,km_pca_dist_00,km_pca_dist_01,km_pca_dist_02,km_pca_dist_03,km_pca_dist_04,km_pca_dist_05,km_pca_dist_06,km_pca_dist_07,meta_lgb_oof,meta_lgb_logit,meta_lr_oof,meta_lr_logit
0,236,1124,0.173529,1104,20,95519.625000,95559.625000,1249.484741,70.235016,665.595337,70.264427,665.591675,13230.237305,-1.0,13230.237305,24682.248047,0.289128,1.810390,18.812820,74.731827,1804.004272,6.222245,0.138450,0.258292,3.0,1.386294,3.0,1.386294,46,0.08,72.869965,72.869965,1.457399,5.432118,39.134056,39.134056,50,0.00,287.325378,287.325378,5.746508,10.636277,49.332207,49.332207,50,0.00,180.226593,180.226593,3.604532,8.125598,48.297508,48.297508,50,0.00,1339.139038,1339.139038,26.782782,129.644119,784.520752,784.520752,...,-0.228989,0.072718,0.132093,-0.026463,-0.230362,-0.143730,-0.224526,-0.122556,-0.011165,0.144867,0.054093,-0.018702,-0.176453,-0.110864,0.120646,0.039116,-0.027804,-0.077748,0.028268,-0.068158,-0.064134,-0.150259,-0.112663,0.048820,0.061549,-0.104185,0.100221,-0.131264,-0.138707,-0.134864,0.025659,-0.151799,0.055772,0.014787,-0.024196,-0.081347,0.033855,0.176032,0.173405,0.094617,-0.073386,0.014633,-0.055600,-0.060842,0.080952,0.239107,14,9.349268,69.663574,82.322296,35.924004,63.682842,81.133194,56.254860,84.748528,58.085808,0.440730,-0.238201,0.476203,-0.095259
1,78,1282,0.057353,1103,179,95358.242188,95716.242188,1358.515137,70.116356,665.609619,70.379593,665.577576,13230.237305,-1.0,13230.237305,24682.248047,0.646781,1.722785,18.812820,74.731827,1804.004272,6.408347,0.138224,0.257869,1.0,0.693147,1.0,0.693147,46,0.08,72.869965,72.869965,1.457399,5.432118,39.134056,39.134056,50,0.00,287.325378,287.325378,5.746508,10.636277,49.332207,49.332207,50,0.00,180.226593,180.226593,3.604532,8.125598,48.297508,48.297508,50,0.00,1339.139038,1339.139038,26.782782,129.644119,784.520752,784.520752,...,0.159665,0.063369,-0.001078,0.138980,0.111497,0.020635,0.099295,0.542352,-0.157361,-0.312392,-0.347760,0.110359,-0.076117,0.052760,-0.050964,-0.078819,-0.081502,0.033048,-0.069944,-0.031090,-0.117855,-0.129142,-0.155985,-0.011017,-0.200283,-0.107933,0.0

## 23. Feature sets

Сохраняем не один набор, а много. Дальше модельный ноутбук сможет выбрать конкретный набор по имени из `feature_sets.json`.

Имена сделаны так, чтобы было понятно, что внутри:

- `raw_clean`: только исходные очищенные признаки;
- `top*_plus_engineered`: топ исходных + все engineered;
- `stable_*`: меньше drifty-фичей;
- `linear_stack_*`: компактные признаки для линейной модели/MLP/stacker;
- `model_meta_*`: наборы с OOF meta-features;
- `heavy_*`: широкие наборы, потенциально сильные, но тяжелее.

In [27]:
raw_cols_set = set(X_train_raw_clean.columns)
eng_cols_set = set(X_train_engineered.columns)

meta_cols = [c for c in X_train_engineered.columns if c.startswith("all_") or c.startswith("combined_") or c.startswith("stable_") or c.startswith("nodrift_")]
pca_svd_cols = [c for c in X_train_engineered.columns if c.startswith("pca_") or c.startswith("qpca_") or c.startswith("svd_") or c.startswith("km_")]
freq_te_peak_cols = [c for c in X_train_engineered.columns if c.startswith("freq_") or c.startswith("logfreq_") or c.startswith("te_") or c.startswith("peak_")]
interaction_cols = [c for c in X_train_engineered.columns if c.startswith("pair_")]
pattern_cols = [c for c in X_train_engineered.columns if "hash" in c or "pattern" in c]
model_meta_cols = [c for c in X_train_engineered.columns if c.startswith("meta_")]
block_cols = [c for c in X_train_engineered.columns if c.startswith("seq_") or c.startswith("ranked_")]

def uniq(seq: Iterable[str]) -> List[str]:
    seen = set(); out = []
    for x in seq:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

feature_sets: Dict[str, List[str]] = {}

# Старые/базовые наборы — чтобы модельные ноутбуки не ломались.
feature_sets["raw_clean"] = list(X_train_raw_clean.columns)
feature_sets["top300_ranked"] = ranked_features[:300]
feature_sets["top500_ranked"] = ranked_features[:500]
feature_sets["top700_ranked"] = ranked_features[:700]
feature_sets["top1000_ranked"] = ranked_features[:1000]
feature_sets["stable_top500_ranked"] = stable_ranked_features[:500]
feature_sets["stable_top700_ranked"] = stable_ranked_features[:700]
feature_sets["nodrift_top700_ranked"] = non_drifty_ranked_features[:700]

feature_sets["raw_clean_plus_meta"] = uniq(feature_sets["raw_clean"] + meta_cols + pattern_cols)
feature_sets["top500_plus_engineered"] = uniq(ranked_features[:500] + list(X_train_engineered.columns))
feature_sets["top700_plus_engineered"] = uniq(ranked_features[:700] + list(X_train_engineered.columns))
feature_sets["top1000_plus_engineered"] = uniq(ranked_features[:1000] + list(X_train_engineered.columns))
feature_sets["all_engineered"] = uniq(list(X_train_raw_clean.columns) + list(X_train_engineered.columns))

# Новые v2 наборы.
feature_sets["v2_meta_only"] = uniq(meta_cols + pattern_cols + block_cols)
feature_sets["v2_pca_svd_stack"] = uniq(meta_cols + pca_svd_cols + freq_te_peak_cols + model_meta_cols)
feature_sets["v2_linear_stack_compact"] = uniq(ranked_features[:250] + meta_cols + pca_svd_cols + freq_te_peak_cols + model_meta_cols)
feature_sets["v2_lgb_top500_meta"] = uniq(ranked_features[:500] + meta_cols + pattern_cols + freq_te_peak_cols + model_meta_cols)
feature_sets["v2_lgb_top700_meta_interactions"] = uniq(ranked_features[:700] + meta_cols + pattern_cols + freq_te_peak_cols + interaction_cols + model_meta_cols)
feature_sets["v2_stable_top700_engineered"] = uniq(stable_ranked_features[:700] + meta_cols + pattern_cols + freq_te_peak_cols + pca_svd_cols + model_meta_cols)
feature_sets["v2_nodrift_top700_engineered"] = uniq(non_drifty_ranked_features[:700] + meta_cols + pattern_cols + freq_te_peak_cols + pca_svd_cols + model_meta_cols)
feature_sets["v2_peak_freq_te"] = uniq(ranked_features[:350] + freq_te_peak_cols + meta_cols + pattern_cols)
feature_sets["v2_interactions_heavy"] = uniq(ranked_features[:500] + interaction_cols + meta_cols + freq_te_peak_cols + model_meta_cols)
feature_sets["v2_model_meta_heavy"] = uniq(ranked_features[:700] + meta_cols + pca_svd_cols + freq_te_peak_cols + interaction_cols + model_meta_cols)
feature_sets["v2_full_beauty"] = uniq(list(X_train_raw_clean.columns) + list(X_train_engineered.columns))

# Синхронизация: оставляем только реально существующие колонки в raw или engineered.
valid_cols = raw_cols_set | eng_cols_set
for name in list(feature_sets.keys()):
    feature_sets[name] = [c for c in feature_sets[name] if c in valid_cols]

feature_sets_summary = pd.DataFrame({
    "feature_set": list(feature_sets.keys()),
    "n_features": [len(v) for v in feature_sets.values()],
}).sort_values("n_features")
display(feature_sets_summary)

with open(OUTPUT_DIR / "feature_sets.json", "w", encoding="utf-8") as f:
    json.dump(feature_sets, f, ensure_ascii=False, indent=2)
feature_sets_summary.to_csv(OUTPUT_DIR / "feature_sets_summary.csv", index=False)
print("Saved feature_sets.json and summary")

,feature_set,n_features
1,top300_ranked,300
2,top500_ranked,500
5,stable_top500_ranked,500
3,top700_ranked,700
6,stable_top700_ranked,700
7,nodrift_top700_ranked,700
13,v2_meta_only,750
4,top1000_ranked,1000
14,v2_pca_svd_stack,1165
0,raw_clean,1360


Saved feature_sets.json and summary


## 24. Сохранение таблиц

Сохраняем в `prepared_data/` в корне проекта. Формат — `parquet`, если доступен; иначе fallback на `pickle`.

Важно: `index` и `target` сохраняются отдельно, чтобы случайно не использовать их как признаки.

In [28]:
def save_table(df: pd.DataFrame, path_stem: Path, index: bool = False) -> Path:
    path_stem.parent.mkdir(parents=True, exist_ok=True)
    parquet_path = path_stem.with_suffix(".parquet")
    pickle_path = path_stem.with_suffix(".pkl")
    try:
        df.to_parquet(parquet_path, index=index)
        print("Saved parquet:", parquet_path, df.shape)
        return parquet_path
    except Exception as e:
        print(f"Could not save parquet for {path_stem.name} ({type(e).__name__}: {e}). Saving pickle instead.")
        df.to_pickle(pickle_path)
        print("Saved pickle:", pickle_path, df.shape)
        return pickle_path

pd.Series(y, name=TARGET_COL).to_csv(OUTPUT_DIR / "y_train.csv", index=False)
train[[ID_COL]].to_csv(OUTPUT_DIR / "train_index.csv", index=False)
test[[ID_COL]].to_csv(OUTPUT_DIR / "test_index.csv", index=False)

save_table(X_train_raw_clean, OUTPUT_DIR / "X_train_raw_clean")
save_table(X_test_raw_clean, OUTPUT_DIR / "X_test_raw_clean")
save_table(X_train_engineered, OUTPUT_DIR / "X_train_engineered")
save_table(X_test_engineered, OUTPUT_DIR / "X_test_engineered")

# Алиасы v2, чтобы было явно видно, что это обновлённый пайплайн.
save_table(X_train_engineered, OUTPUT_DIR / "X_train_engineered_v2")
save_table(X_test_engineered, OUTPUT_DIR / "X_test_engineered_v2")
save_table(train_model_meta, OUTPUT_DIR / "X_train_model_meta_features")
save_table(test_model_meta, OUTPUT_DIR / "X_test_model_meta_features")

# Сохраняем списки колонок отдельно для удобства.
with open(OUTPUT_DIR / "raw_clean_columns.json", "w", encoding="utf-8") as f:
    json.dump(list(X_train_raw_clean.columns), f, ensure_ascii=False, indent=2)
with open(OUTPUT_DIR / "engineered_columns.json", "w", encoding="utf-8") as f:
    json.dump(list(X_train_engineered.columns), f, ensure_ascii=False, indent=2)

print("All saved to", OUTPUT_DIR)

Saved parquet: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/prepared_data/X_train_raw_clean.parquet (247972, 1360)
Saved parquet: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/prepared_data/X_test_raw_clean.parquet (106274, 1360)
Saved parquet: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/prepared_data/X_train_engineered.parquet (247972, 4089)
Saved parquet: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/prepared_data/X_test_engineered.parquet (106274, 4089)
Saved parquet: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/prepared_data/X_train_engineered_v2.parquet (247972, 4089)
Saved parquet: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/prepared_data/X_test_engineered_v2.parquet (106274, 4089)
Saved parquet: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/prepared_data/X_train_model_meta_features.parquet (247972, 4)
Saved parquet: /Users/pinta/hse/mlproject/ml-pro

## 25. Cобрать матрицу по имени feature set

Эта функция нужна и для benchmark внутри ноутбука, и как пример для будущих modeling notebooks.

In [29]:
def get_matrix_for_feature_set(name: str) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    cols = feature_sets[name]
    raw_cols = [c for c in cols if c in X_train_raw_clean.columns]
    eng_cols = [c for c in cols if c in X_train_engineered.columns]

    tr_parts = []
    te_parts = []
    if raw_cols:
        tr_parts.append(X_train_raw_clean[raw_cols])
        te_parts.append(X_test_raw_clean[raw_cols])
    if eng_cols:
        tr_parts.append(X_train_engineered[eng_cols])
        te_parts.append(X_test_engineered[eng_cols])

    Xtr = pd.concat(tr_parts, axis=1).loc[:, lambda d: ~d.columns.duplicated()].copy()
    Xte = pd.concat(te_parts, axis=1).reindex(columns=Xtr.columns, fill_value=0)
    return Xtr, Xte, Xtr.columns.tolist()

for name in ["raw_clean", "top500_plus_engineered", "v2_lgb_top700_meta_interactions", "v2_full_beauty"]:
    if name in feature_sets:
        Xtmp, _, cols = get_matrix_for_feature_set(name)
        print(name, Xtmp.shape)
        del Xtmp
        gc.collect()

raw_clean (247972, 1360)
top500_plus_engineered (247972, 4589)
v2_lgb_top700_meta_interactions (247972, 4195)
v2_full_beauty (247972, 5449)


## 26. Быстрый, но честный benchmark feature sets

Это не финальное обучение. Цель — понять, какие наборы признаков дальше скармливать сильному ensemble notebook.

Используем один LightGBM-конфиг и одинаковые folds. Сохраняем `feature_set_benchmark_lgb_v2.csv`.

In [30]:
def benchmark_lgb_feature_set(
    name: str,
    max_boost_rounds: int = 3000,
    early_stopping_rounds: int = 120,
) -> Dict[str, float]:
    if not HAS_LIGHTGBM:
        print("LightGBM not installed; skip", name)
        return {"feature_set": name, "n_features": len(feature_sets[name]), "mean_auc": np.nan, "std_auc": np.nan}

    Xtr, _, cols = get_matrix_for_feature_set(name)
    pos = y.sum(); neg = len(y) - pos
    params = {
        "objective": "binary",
        "metric": "auc",
        "learning_rate": 0.025,
        "num_leaves": 48,
        "min_data_in_leaf": 80,
        "feature_fraction": 0.80,
        "bagging_fraction": 0.85,
        "bagging_freq": 1,
        "lambda_l1": 0.1,
        "lambda_l2": 20.0,
        "scale_pos_weight": float(neg / max(pos, 1)),
        "verbosity": -1,
        "num_threads": N_JOBS,
        "seed": RANDOM_STATE,
        "force_col_wise": True,
    }

    aucs = []
    best_iters = []
    for fold, (tr_idx, va_idx) in enumerate(cv_splits):
        print(f"Benchmark {name} fold {fold}")
        dtr = lgb.Dataset(Xtr.iloc[tr_idx], label=y[tr_idx], free_raw_data=False)
        dva = lgb.Dataset(Xtr.iloc[va_idx], label=y[va_idx], reference=dtr, free_raw_data=False)
        model = lgb.train(
            params,
            dtr,
            num_boost_round=max_boost_rounds,
            valid_sets=[dva],
            valid_names=["valid"],
            callbacks=[lgb.early_stopping(early_stopping_rounds, verbose=False), lgb.log_evaluation(250)],
        )
        pred = model.predict(Xtr.iloc[va_idx], num_iteration=model.best_iteration)
        auc = roc_auc_score(y[va_idx], pred)
        aucs.append(float(auc))
        best_iters.append(int(model.best_iteration or max_boost_rounds))
        print(f"{name} fold={fold} auc={auc:.6f} best_iter={model.best_iteration}")
        del dtr, dva, model
        gc.collect()

    del Xtr
    gc.collect()
    return {
        "feature_set": name,
        "n_features": len(cols),
        "mean_auc": float(np.mean(aucs)),
        "std_auc": float(np.std(aucs)),
        "min_auc": float(np.min(aucs)),
        "max_auc": float(np.max(aucs)),
        "mean_best_iter": float(np.mean(best_iters)),
    }

benchmark_results = []
if RUN_BENCHMARK:
    benchmark_names = [
        "raw_clean",
        "top500_ranked",
        "top500_plus_engineered",
        "top700_plus_engineered",
        "v2_lgb_top500_meta",
        "v2_lgb_top700_meta_interactions",
        "v2_stable_top700_engineered",
        "v2_nodrift_top700_engineered",
        "v2_peak_freq_te",
        "v2_pca_svd_stack",
        "v2_linear_stack_compact",
        "v2_model_meta_heavy",
        "v2_full_beauty",
    ]
    benchmark_names = [n for n in benchmark_names if n in feature_sets]
    if BENCHMARK_MAX_FEATURE_SETS is not None:
        benchmark_names = benchmark_names[:BENCHMARK_MAX_FEATURE_SETS]

    t0 = time.time()
    for name in benchmark_names:
        try:
            res = benchmark_lgb_feature_set(name)
            benchmark_results.append(res)
            pd.DataFrame(benchmark_results).sort_values("mean_auc", ascending=False).to_csv(
                OUTPUT_DIR / "feature_set_benchmark_lgb_v2_partial.csv", index=False
            )
            display(pd.DataFrame(benchmark_results).sort_values("mean_auc", ascending=False))
        except Exception as e:
            print("Benchmark failed for", name, type(e).__name__, e)
            benchmark_results.append({"feature_set": name, "n_features": len(feature_sets[name]), "mean_auc": np.nan, "std_auc": np.nan, "error": repr(e)})
    print("Benchmark total time", round(time.time() - t0, 1), "s")
else:
    print("RUN_BENCHMARK=False")

benchmark_df = pd.DataFrame(benchmark_results)
if len(benchmark_df):
    benchmark_df = benchmark_df.sort_values("mean_auc", ascending=False)
    benchmark_df.to_csv(OUTPUT_DIR / "feature_set_benchmark_lgb_v2.csv", index=False)
    display(benchmark_df)
    print("Saved:", OUTPUT_DIR / "feature_set_benchmark_lgb_v2.csv")

Benchmark raw_clean fold 0
[250]	valid's auc: 0.633294
raw_clean fold=0 auc=0.635081 best_iter=185
Benchmark raw_clean fold 1
[250]	valid's auc: 0.648287
raw_clean fold=1 auc=0.649300 best_iter=225
Benchmark raw_clean fold 2
[250]	valid's auc: 0.616914
raw_clean fold=2 auc=0.617345 best_iter=284
Benchmark raw_clean fold 3
[250]	valid's auc: 0.630659
raw_clean fold=3 auc=0.631897 best_iter=240
Benchmark raw_clean fold 4
[250]	valid's auc: 0.641044
raw_clean fold=4 auc=0.641866 best_iter=226


,feature_set,n_features,mean_auc,std_auc,min_auc,max_auc,mean_best_iter
0,raw_clean,1360,0.635098,0.010704,0.617345,0.6493,232.0


Benchmark top500_ranked fold 0
[250]	valid's auc: 0.630574
top500_ranked fold=0 auc=0.632361 best_iter=209
Benchmark top500_ranked fold 1
[250]	valid's auc: 0.653639
top500_ranked fold=1 auc=0.655761 best_iter=287
Benchmark top500_ranked fold 2
[250]	valid's auc: 0.630382
top500_ranked fold=2 auc=0.630724 best_iter=252
Benchmark top500_ranked fold 3
[250]	valid's auc: 0.63584
top500_ranked fold=3 auc=0.637485 best_iter=168
Benchmark top500_ranked fold 4
[250]	valid's auc: 0.642728
top500_ranked fold=4 auc=0.644395 best_iter=239


,feature_set,n_features,mean_auc,std_auc,min_auc,max_auc,mean_best_iter
1,top500_ranked,500,0.640145,0.009147,0.630724,0.655761,231.0
0,raw_clean,1360,0.635098,0.010704,0.617345,0.649300,232.0


Benchmark top500_plus_engineered fold 0
[250]	valid's auc: 0.701306
top500_plus_engineered fold=0 auc=0.702306 best_iter=162
Benchmark top500_plus_engineered fold 1
[250]	valid's auc: 0.700152
top500_plus_engineered fold=1 auc=0.704780 best_iter=135
Benchmark top500_plus_engineered fold 2
[250]	valid's auc: 0.698694
top500_plus_engineered fold=2 auc=0.699760 best_iter=211
Benchmark top500_plus_engineered fold 3
top500_plus_engineered fold=3 auc=0.707336 best_iter=79
Benchmark top500_plus_engineered fold 4
[250]	valid's auc: 0.687983
top500_plus_engineered fold=4 auc=0.689698 best_iter=137


...

Benchmark v2_full_beauty fold 0
v2_full_beauty fold=0 auc=0.697274 best_iter=102
Benchmark v2_full_beauty fold 1
[250]	valid's auc: 0.704113
v2_full_beauty fold=1 auc=0.704980 best_iter=200
Benchmark v2_full_beauty fold 2
[250]	valid's auc: 0.69576
v2_full_beauty fold=2 auc=0.697678 best_iter=175
Benchmark v2_full_beauty fold 3
[250]	valid's auc: 0.701763
v2_full_beauty fold=3 auc=0.702942 best_iter=212
Benchmark v2_full_beauty fold 4
[250]	valid's auc: 0.686633
v2_full_beauty fold=4 auc=0.689406 best_iter=167


,feature_set,n_features,mean_auc,std_auc,min_auc,max_auc,mean_best_iter
8,v2_peak_freq_te,1361,0.704447,0.007076,0.696042,0.713817,278.8
9,v2_pca_svd_stack,1165,0.702001,0.007667,0.689298,0.712599,136.8
2,top500_plus_engineered,4589,0.700776,0.006085,0.689698,0.707336,144.8
10,v2_linear_stack_compact,1415,0.700512,0.007603,0.687477,0.710530,119.4
7,v2_nodrift_top700_engineered,1869,0.700050,0.007462,0.686818,0.709171,200.0
4,v2_lgb_top500_meta,1515,0.699564,0.007325,0.687798,0.709421,122.6
12,v2_full_beauty,5449,0.698456,0.005415,0.689406,0.704980,171.2
3,top700_plus_engineered,4789,0.698379,0.010816,0.684117,0.713455,224.6
6,v2_stable_top700_engineered,1869,0.698172,0.008060,0.687225,0.710002,136.8
5,v2_lgb_top700_meta_interactions,4195,0.696840,0.011274,0.677955,0.707236,148.8


Benchmark total time 3401.7 s


,feature_set,n_features,mean_auc,std_auc,min_auc,max_auc,mean_best_iter
8,v2_peak_freq_te,1361,0.704447,0.007076,0.696042,0.713817,278.8
9,v2_pca_svd_stack,1165,0.702001,0.007667,0.689298,0.712599,136.8
2,top500_plus_engineered,4589,0.700776,0.006085,0.689698,0.707336,144.8
10,v2_linear_stack_compact,1415,0.700512,0.007603,0.687477,0.710530,119.4
7,v2_nodrift_top700_engineered,1869,0.700050,0.007462,0.686818,0.709171,200.0
4,v2_lgb_top500_meta,1515,0.699564,0.007325,0.687798,0.709421,122.6
12,v2_full_beauty,5449,0.698456,0.005415,0.689406,0.704980,171.2
3,top700_plus_engineered,4789,0.698379,0.010816,0.684117,0.713455,224.6
6,v2_stable_top700_engineered,1869,0.698172,0.008060,0.687225,0.710002,136.8
5,v2_lgb_top700_meta_interactions,4195,0.696840,0.011274,0.677955,0.707236,148.8


Saved: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/prepared_data/feature_set_benchmark_lgb_v2.csv
